<a href="https://colab.research.google.com/github/Geauga/Proteina-Complexa-dev/blob/main/coLab/Proteina_Complexa_coLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  # Section 0: Global Configuration & Target Initialization (State Saved; Re-run Optional After Restart)

<details>
<summary><font size="5"><b>⚙️ Expand: Global Configuration & Workflow Guidelines</b></font></summary>

<br>

### 💡 ARCHITECTURE OVERVIEW
This notebook employs a decoupled "Front-end UI / Back-end Engine" architecture. Configure all parameters exclusively in the single **Section 0** dashboard, which persists data to Google Drive, permitting uninterrupted execution across kernel restarts.

### 🛑 CRITICAL NOTICE: MANDATORY KERNEL RESTART
Upon completing **Section 1** (Environment Configuration), a Colab runtime restart is strictly required to apply core library upgrades (JAX/Flax/CUDA).
**ACTION:** Navigate to `Runtime -> Restart session` prior to subsequent executions. Failure to comply will result in pipeline termination.

### 📦 ZERO-CONFIGURATION ASSETS
The demonstration structure (`GFP_1_10.pdb`) is pre-integrated. For custom targets, upload PDB files directly to the `Proteina-Complexa/targets/` directory within Google Drive, or utilize the integrated upload toggle within Section 0.

---
### 🔄 EXECUTION WORKFLOWS

**Phase I: Universal Initialization**
1. **Global Configuration:** Adjust all parameters (Standard, Auto-Pilot, Geometric) in the **Section 0** UI dashboard and execute the cell to register the variables persistently.
2. **Environment Initialization:** Expand and execute **Section 1**. *Note: Initial Compilation and Asset Synchronization may require 10–20 minutes; this is an intensive background process and does not indicate a System Deadlock. Opting for Google Drive persistent backup (approx. 5GB) stores stabilized binaries, enabling subsequent sessions to bypass these stages even after a Runtime deletion.*
3. ⚠️ **RESTART SESSION** (`Runtime -> Restart session`).
4. **Target Pre-processing:** *(Re-running Section 0 post-restart is optional)*. Expand and execute **Section 2** to parse and prepare the target structure.

**Phase II: Divergent Execution Pathways (Choose One)**
* **Workflow A: Standard Linear Pipeline.** Expand **Section 3**. Execute the standard generation engine, followed sequentially by the diagnostic, scanning, and packaging modules.
* **Workflow B: High-Throughput Auto-Pilot.** Expand **Section 4**. Execute the autonomous engine to continuously generate, evaluate, and log candidates until the predefined success threshold is attained.
* **Workflow C: Geometric Surface Scanning.** Expand **Section 5**. Execute the spatial algorithmic engine to dynamically identify exposed surface topologies and process targeted hotspot patches.
* **Workflow D: Modular Decoupled Execution.** Expand **Section 6**. Independently trigger pure generation or isolated structural evaluation modules for granular VRAM control and architectural debugging.

</details>

In [ ]:
# @title 🎯 Section_0_Global_Configuration_Dashboard { display-mode: "form" }
# @markdown Complete this dashboard **ONCE**. All settings are persistently saved to your Google Drive.

import os, json, shutil, sys
from pathlib import Path
from IPython.display import display, HTML

# ==========================================
# 0. Infrastructure & Google Drive Mount
# ==========================================
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print("🔄 Mounting Google Drive for persistent storage...")
    drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

# --- Source Repository Detection and Cloning ---
if not os.path.exists('/content/drive/MyDrive/Proteina-Complexa/pyproject.toml'):
    print(f">>> Source repository not detected. Executing initial clone to {ROOT_DIR}...")
    !git clone https://github.com/Geauga/Proteina-Complexa-coLab {ROOT_DIR}
else:
    print(f">>> Source repository already exists at {ROOT_DIR}. Skipping clone step.")

TARGETS_DIR = BASE_DIR / 'targets'
TARGETS_DIR.mkdir(parents=True, exist_ok=True)
SETTING_FILE = BASE_DIR / 'internal_settings.json'

# ==========================================
# --- Configuration for Environment Setup & Cache (Cell 3) ---
# ==========================================
# @markdown ### 💾 Environment & Cache Settings
# @markdown **[Accelerate Startup]** Enable Google Drive backup for the build cache?
# @markdown > *Note: The very first run will be slightly slower (1-2 mins) to package the backup; subsequent startups will be lightning fast.*
enable_cache_backup = True # @param {type:"boolean"}

# ==========================================
# --- Configuration for Target Setup & Processing (Cell 4) ---
# ==========================================
# @markdown ---
# @markdown ### 📁 Target File Selection
UPLOAD_NEW_PDB = False #@param {type:"boolean"}
pdb_file_name = "GFP_1_10.pdb" #@param {type:"string"}

# @markdown ### 🧬 Target Definition
task_name = "GFP" #@param {type:"string"}
target_chains = "A" #@param {type:"string"}

# @markdown ### 🎯 Interaction Hotspots (Comma separated)
hotspots_input = "39,40,41,73,74,75,75,77,78,199,200,201,202,203,204" #@param {type:"string"}

# @markdown ### 📏 Binder Peptide Length
binder_length_min = 30 #@param {type:"integer"}
binder_length_max = 60 #@param {type:"integer"}

# ==========================================
# --- Pipeline Configuration (Standard Gen & Auto-Pilot - Cell 5 & 9) ---
# ==========================================
num_designs = 360 #@param {type:"integer"}
gen_batch_size = 18 #@param {type:"integer"}
run_filter = True #@param {type:"boolean"}
run_evaluate = True #@param {type:"boolean"}
run_analyze = True #@param {type:"boolean"}
target_threshold = 0.80 #@param {type:"number"}
target_success_count = 10 #@param {type:"integer"}
designs_per_loop = 18 #@param {type:"integer"}
max_iterations = 50 #@param {type:"integer"}
batch_size = 18 #@param {type:"integer"}

# ==========================================
# --- Geometric Surface Scanning Config (Cell 10) ---
# ==========================================
# @markdown ---
# @markdown ### 📐 Geometric Surface Scanning
anchor_residues = "39,40,41,73,74,75,75,77,78,199,200,201,202,203,204" #@param {type:"string"}
scan_margin = 10.0 #@param {type:"number"}
local_patch_radius = 8.0 #@param {type:"number"}
designs_per_patch = 48 #@param {type:"integer"}
scan_batch_size = 18 #@param {type:"integer"}

# ==========================================
# --- Automated Multi-Dimensional Surface Scanning (Cell 11) ---
# ==========================================
# @markdown ---
# @markdown ### 🔬 Automated Groove & Pocket Discovery
scan_shallow_groove = True #@param {type:"boolean"}
scan_deep_pocket = True #@param {type:"boolean"}
scan_hydrophobic_patch = True #@param {type:"boolean"}
scan_charged_patch = True #@param {type:"boolean"}



# ==========================================
# Execution Logic: Validation & Persistence
# ==========================================

final_pdb_name = pdb_file_name

if UPLOAD_NEW_PDB:
    from google.colab import files
    display(HTML('<p style="font-size:16px; font-weight:bold; color:#E32636;">👉 PLEASE CLICK "CHOOSE FILES" TO UPLOAD:</p>'))
    uploaded = files.upload()
    if uploaded:
        for filename in uploaded.keys():
            shutil.move(filename, TARGETS_DIR / filename)
            final_pdb_name = filename
            break

# Bundle settings
settings = {
    "enable_cache_backup": enable_cache_backup,
    "pdb_file": final_pdb_name,
    "task_name": task_name,
    "target_chains": target_chains,
    "hotspots_input": hotspots_input,
    "binder_length_min": binder_length_min,
    "binder_length_max": binder_length_max,
    "num_designs": num_designs,
    "gen_batch_size": gen_batch_size,
    "run_filter": run_filter,
    "run_evaluate": run_evaluate,
    "run_analyze": run_analyze,
    "target_threshold": target_threshold,
    "target_success_count": target_success_count,
    "designs_per_loop": designs_per_loop,
    "max_iterations": max_iterations,
    "batch_size": batch_size,
    "anchor_residues": anchor_residues,
    "scan_margin": scan_margin,
    "local_patch_radius": local_patch_radius,
    "designs_per_patch": designs_per_patch,

}

# Persistent Save
with open(SETTING_FILE, 'w') as f:
    json.dump(settings, f, indent=4)

print(f"\n✅ SETTINGS REGISTERED")
print(f"Task Name  : {task_name}")
print(f"Target PDB : {final_pdb_name} (Chain: {target_chains})")

print("💾 Data saved. The backend engines will seamlessly inherit these unified configurations.")

>>> Source repository already exists at /content/drive/MyDrive/Proteina-Complexa. Skipping clone step.

✅ SETTINGS REGISTERED
Task Name  : GFP
Target PDB : GFP_1_10.pdb (Chain: A)
💾 Data saved. The backend engines will seamlessly inherit these unified configurations.


# Section 1: Environment Configuration and Dependency Management (Restart Session after All 4 Cells are Finished after the First Run)

In [ ]:
# Cell_1_Environment_Initialization.py
# Requirement: Upgrade essential logging and serialization dependencies (wandb, protobuf) for the pipeline.

# ==========================================
# --- Environment Base Dependencies ---
# ==========================================
!pip install --upgrade wandb protobuf

# Purpose: Ensure the base environment has the latest required versions of Weights & Biases (wandb) for experiment tracking and protocol buffers (protobuf) for model serialization.
# Upstream Code: The first cell of the provided Colab notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:30 EDT.
# Changed Lines:
# * Designated the cell as "Cell 1" in the header.
# * Appended the standard metadata tracking block.

  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 111.6 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.23.1
    Uninstalling wandb-0.23.1:
      Successfully uninstalled wandb-0.23.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
proteinfoundation 1.0.0 requires wandb==0.23.1, but you have wandb 0.25.1 which is incompatible.


In [ ]:
# Cell_2_Mount_Google_Drive.py
# Requirement: Mount Google Drive to the Colab instance to enable access to persistent storage for the Proteina-Complexa project files.

from google.colab import drive
drive.mount('/content/drive')

# Purpose: Connect the Google Colab virtual machine to the user's Google Drive, allowing the pipeline to read configurations, load models, and save generated protein structures persistently.
# Upstream Code: The original Google Drive mounting cell (formerly labeled as Cell 1) from the uploaded notebook.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 08:35 EDT.
# Changed Lines:
# * Designated the cell as "Cell 2" in the header.
# * Appended the standard metadata tracking block.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell_2b_environment_setup_engine.py
import os


BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")

%cd {ROOT_DIR}

def setup_environment():
    print(">>> Initializing installation and environment configuration...")

    # 1. Base Installation
    !pip install uv
    !uv pip install --system --index-strategy unsafe-best-match -e .

    # 2. Consolidated Dependency Installation
    !uv pip install --system git+https://github.com/RosettaCommons/atomworks.git
    !uv pip install --system torch-scatter --no-build-isolation
    !uv pip install --system dm-haiku openbabel-wheel graphein e3nn

    # 3. Deploy Foldseek Binary
    if not os.path.exists(".venv/bin/foldseek"):
        !mkdir -p .venv/bin
        !wget -q -nc https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
        !tar xzf foldseek-linux-avx2.tar.gz
        !mv foldseek/bin/foldseek .venv/bin/ && chmod +x .venv/bin/foldseek
        !rm -rf foldseek-linux-avx2.tar.gz foldseek/

    # 4. Create sc Script Mock
    !mkdir -p env/docker/internal/
    !echo -e '#!/bin/bash\necho "0.000"' > env/docker/internal/sc && chmod +x env/docker/internal/sc

    # 5. Initialization and Model Download
    !complexa init uv
    !source env.sh && complexa download --complexa-all
    !source env.sh && complexa download --all

    # 6. Path Correction (.env)
    code_path = ROOT_DIR + "/"
    data_path = os.path.join(ROOT_DIR, "assets/")
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE, "r") as f: lines = f.readlines()
        with open(ENV_FILE, "w") as f:
            for line in lines:
                if line.startswith("LOCAL_CODE_PATH="): f.write(f"LOCAL_CODE_PATH={code_path}\n")
                elif line.startswith("LOCAL_DATA_PATH="): f.write(f"LOCAL_DATA_PATH={data_path}\n")
                else: f.write(line)

    # 7. Task Completion: Conditional Backup (Reads enable_cache_backup from Cell 2a)
    if enable_cache_backup:
        if not os.path.exists(UV_CACHE_TAR):
            print(">>> User opted in for cache backup. Extracting and backing up underlying C++ build cache to Drive...")
            !mkdir -p {BACKUP_DIR}
            !mkdir -p ~/.cache/uv
            !cd ~ && tar -cf {UV_CACHE_TAR} .cache/uv
            print(">>> Cache backup complete. Future reboots will achieve rapid recovery.")
        else:
            print(">>> Existing cache backup detected in Drive. Skipping redundant packaging.")
    else:
        print(">>> Cache backup bypassed per user configuration.")

# ====================================================================
# --- Main Logic: Inject cache first, then execute configuration ---
# ====================================================================
if os.path.exists(UV_CACHE_TAR):
    print(">>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...")
    !mkdir -p ~/.cache
    !tar -xf {UV_CACHE_TAR} -C ~
    print(">>> Cache injection complete. Preparing environment reconstruction...")
    setup_environment()
else:
    print(">>> No cache detected. Commencing initial heavy compilation (please wait)...")
    setup_environment()

print(">>> Environment ready. Initiating final validation...")
!source env.sh && complexa validate design configs/search_binder_local_pipeline.yaml

/content/drive/MyDrive/Proteina-Complexa
>>> 🛡️ Cloud uv build cache detected. Performing extraction and overwrite...
>>> Cache injection complete. Preparing environment reconstruction...
>>> Initializing installation and environment configuration...
Using Python 3.12.13 environment at: /usr
Resolved 138 packages in 2.08s
Prepared 1 package in 690ms
Uninstalled 5 packages in 162ms
Installed 5 packages in 35ms
 - biotite==1.4.0
 + biotite==0.41.2
 - numpy==2.4.4
 + numpy==1.26.4
 ~ proteinfoundation==1.0.0 (from file:///content/drive/MyDrive/Proteina-Complexa)
 - scipy==1.17.1
 + scipy==1.12.0
 - wandb==0.25.1
 + wandb==0.23.1
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 1.66s
Uninstalled 2 packages in 22ms
Installed 2 packages in 28ms
 - biotite==0.41.2
 + biotite==1.4.0
 - scipy==1.12.0
 + scipy==1.17.1
Using Python 3.12.13 environment at: /usr
Checked 1 package in 96ms
Using Python 3.12.13 environment at: /usr
Checked 4 packages in 111ms

  env.sh already exists.

In [ ]:
# Cell_3_Repository_Clone_and_JAX_Patch.py
# Requirement: Detect and clone the Proteina-Complexa repository if absent. Upgrade core machine learning libraries (JAX, Flax, Haiku), and dynamically patch deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.

# ==========================================
# --- Repository Initialization & Environment Patching ---
# ==========================================
import os
import sys
import subprocess
from pathlib import Path

# --- Path Configuration ---
ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"
UV_CACHE_TAR = os.path.join(BACKUP_DIR, "uv_build_cache.tar")
ENV_FILE = os.path.join(ROOT_DIR, ".env")


# Align Paths
BASE_DIR = Path(ROOT_DIR)
COLABDESIGN_DIR = BASE_DIR / 'community_models/colabdesign'

def run_command(command_list):
    print(f"🚀 Executing: {' '.join(command_list)[:80]}...")
    process = subprocess.Popen(command_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout: print(line, end='', flush=True)
    process.wait()

# 1. Upgrade core dependencies (tailored for GPU environment)
print("\n[Stage 1/3] Upgrading JAX and core ecosystem libraries...")
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "jax[cuda12]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"])
run_command([sys.executable, "-m", "pip", "install", "--upgrade", "flax", "dm-haiku", "chex", "optax"])

# 2. Patch JAX underlying NumPy compatibility (for Python 3.12+)
print("\n[Stage 2/3] Patching JAX and NumPy conflicts...")
literals_path = Path('/usr/local/lib/python3.12/dist-packages/jax/_src/literals.py')
if literals_path.exists():
    code = literals_path.read_text()
    if ", copy=copy" in code:
        literals_path.write_text(code.replace(", copy=copy", ""))
        print("✅ JAX copy=copy patch applied successfully!")

# 3. Deep patching for ColabDesign source code
print("\n[Stage 3/3] Patching ColabDesign source code compatibility...")
if COLABDESIGN_DIR.exists():
    for filepath in COLABDESIGN_DIR.rglob('*.py'):
        code = filepath.read_text()
        new_code = code.replace('jax.lib.xla_bridge.get_backend()', 'jax.extend.backend.get_backend()') \
                       .replace('.live_buffers()', '.live_arrays()') \
                       .replace('jax.tree_map', 'jax.tree_util.tree_map') \
                       .replace('jax.tree_flatten', 'jax.tree_util.tree_flatten') \
                       .replace('jax.tree_unflatten', 'jax.tree_util.tree_unflatten') \
                       .replace('jax.tree_leaves', 'jax.tree_util.tree_leaves')
        if '@jax.util.wraps' in new_code:
            new_code = new_code.replace('@jax.util.wraps(fun, docstr=docstr)', '@functools.wraps(fun)') \
                               .replace('@jax.util.wraps(fun)', '@functools.wraps(fun)')
            if 'import functools' not in new_code: new_code = 'import functools\n' + new_code
        if new_code != code: filepath.write_text(new_code)
    print("✅ ColabDesign patching completed.")
else:
    print("⚠️ ColabDesign directory not found. Skipping Stage 3 patching.")

print("\n✨ Environment patching is complete. You may now run the evaluation cells below.")

# Purpose: Integrate repository cloning with machine learning dependency upgrades. Patches deprecated syntax in JAX and ColabDesign to ensure compatibility with Python 3.12+ and modern JAX versions.
# Upstream Code: User provided combined logic for Git cloning and JAX/ColabDesign patching.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-01 09:42 EDT.
# Changed Lines:
# * Consolidated duplicate imports and harmonized base directory variables natively.
# * Added an `else` safeguard block for Stage 3 in case the ColabDesign directory is absent at runtime.


[Stage 1/3] Upgrading JAX and core ecosystem libraries...
🚀 Executing: /usr/bin/python3 -m pip install --upgrade jax[cuda12] -f https://storage.googlea...
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphein 1.7.8 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
proteinfoundation 1.0.0 requires scipy==1.12.0, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have

# Section 2: Target Specification and Structural Pre-processing

In [ ]:

# Cell_4b_PDB_Processing_and_YAML_Configuration_Generation.py
# Requirement: Process user PDB and dynamically build the target configuration yaml. Scan the 'coLab' directory and copy all .pdb files to the 'targets' directory, skipping existing files.

import os, json, yaml, shutil, time, sys
from pathlib import Path

print("🔍 Initializing Target Processing Engine...")

# ---------------------------------------------------------
# 1. Load Persistent Configuration from Google Drive
# ---------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

SETTING_FILE = Path('/content/drive/MyDrive/Proteina-Complexa/internal_settings.json')
if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found.")
    print("💡 Action: Please scroll up to Section 0, complete the form, and run the Global Dashboard cell first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

# Extract essential variables dynamically
pdb_file = cfg['pdb_file']
task_name = cfg['task_name']
target_chains = cfg['target_chains']
hotspots_input = str(cfg['hotspots_input'])
binder_length_min = int(cfg['binder_length_min'])
binder_length_max = int(cfg['binder_length_max'])

print(f"✅ Loaded persistent settings for task: [{task_name}]")

# ---------------------------------------------------------
# 2. Path Setup & Verification
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGETS_INPUT_DIR = BASE_DIR / 'targets'
RUN_DATA_DIR = BASE_DIR / 'assets' / 'target_data' / task_name
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'
DEMO_DIR = BASE_DIR / 'coLab'

TARGETS_INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- [Logic Add: Batch Demo File Auto-Initialization] ---
print("📦 Synchronizing demo PDB files from repository...")
if DEMO_DIR.exists():
    for pdb_item in DEMO_DIR.glob('*.pdb'):
        dest_item = TARGETS_INPUT_DIR / pdb_item.name
        if not dest_item.exists():
            shutil.copy(pdb_item, dest_item)
            print(f"  -> Copied: {pdb_item.name}")
        else:
            print(f"  -> Skipped (Already exists): {pdb_item.name}")
else:
    print(f"⚠️ Warning: Source directory '{DEMO_DIR.name}' not found.")

raw_pdb_path = TARGETS_INPUT_DIR / pdb_file
fixed_pdb_path = RUN_DATA_DIR / f"{task_name}_fixed.pdb"

if not raw_pdb_path.exists():
    print(f"❌ Error: Required PDB file '{pdb_file}' not found in '{TARGETS_INPUT_DIR}'.")
    sys.exit(1)

# ---------------------------------------------------------
# 3. PDB Parsing & Missing Gap Stitching
# ---------------------------------------------------------
print(f"⚙️ Parsing raw structure '{pdb_file}' and stitching missing residues...")

valid_res_nums = set()
with open(raw_pdb_path, 'r') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA' and line[21] == target_chains:
            try: valid_res_nums.add(int(line[22:26].strip()))
            except: continue

with open(raw_pdb_path, 'r') as f_in, open(fixed_pdb_path, 'w') as f_out:
    for line in f_in:
        if line.startswith('ATOM') and line[21] == target_chains:
            try:
                if int(line[22:26].strip()) in valid_res_nums: f_out.write(line)
            except: pass
        elif line.startswith('TER'): f_out.write(line)

res_nums = sorted(list(valid_res_nums))
ranges = []
if res_nums:
    start = prev = res_nums[0]
    for n in res_nums[1:]:
        if n == prev + 1: prev = n
        else:
            ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")
            start = prev = n
    ranges.append(f"{target_chains}{start}-{prev}" if start!=prev else f"{target_chains}{start}")

# ---------------------------------------------------------
# 4. YAML Configuration Generation
# ---------------------------------------------------------
hotspots_list = [int(x.strip()) for x in hotspots_input.split(',') if x.strip().isdigit()]

yaml_data = {'target_dict_cfg': {
    task_name: {
        'target_input': ','.join(ranges),
        'binder_length': [binder_length_min, binder_length_max],
        'target_chains': [target_chains],
        'pdb_id': task_name,
        'source': task_name,
        'target_filename': task_name + "_fixed",
        'hotspot_residues': [f"{target_chains}{i}" for i in hotspots_list]
    }
}}

YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
if YAML_PATH.exists():
    backup_name = f"targets_dict_backup_{int(time.time())}.yaml"
    shutil.copy(YAML_PATH, YAML_PATH.parent / backup_name)
    print(f"💾 Existing config safely backed up as: {backup_name}")

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"🎯 SUCCESS: Target configuration generated dynamically for '{task_name}'. Ready for generation phase!")

# Purpose: Process PDB, build YAML configuration, and batch copy all demo PDB files from the repository clone to the working targets directory without overwriting existing files.
# Upstream Code: Configuration read from Section 0 (Global Dashboard).
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 14:53 EDT.
# Changed Lines:
# * Line 38: Replaced single DEMO_PDB_SOURCE file path with DEMO_DIR directory path.
# * Lines 44-53: Replaced the single file copy logic with a for-loop iterating over DEMO_DIR.glob('*.pdb'), checking dest_item.exists() before executing shutil.copy().

🔍 Initializing Target Processing Engine...
✅ Loaded persistent settings for task: [GFP]
📦 Synchronizing demo PDB files from repository...
  -> Skipped (Already exists): GFP_1_10.pdb
⚙️ Parsing raw structure 'GFP_1_10.pdb' and stitching missing residues...
⚠️ Warning: Native baseline 'GFP_11.pdb' not found in '/content/drive/MyDrive/Proteina-Complexa/targets'. Skipping native fix.
💾 Existing config safely backed up as: targets_dict_backup_1775315522.yaml
🎯 SUCCESS: Target configuration generated dynamically for 'GFP'. Ready for generation phase!


# Section 3: Evaluating Natural Peptide

In [76]:

# =================================================================
# 🎯 GFP 天然底物基准评估引擎 + 详细数据面板 + 3D渲染全自动流水线
# =================================================================

import os, sys, subprocess, shutil, gc, torch, time
import pandas as pd
from pathlib import Path
import py3Dmol

# --- [配置区] ---
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
TARGET_PDB_NAME = "GFP_1_10.pdb"
BINDER_PDB_NAME = "GFP_11.pdb"  # 我们已经确认这里面是 10AA 核心序列
TARGET_TASK = "GFP"
RUN_NAME = "Run_Native_Baseline"
JOB_ID = 0

# --- [1. 路径预检] ---
target_pdb_path = BASE_DIR / "targets" / TARGET_PDB_NAME
binder_pdb_path = BASE_DIR / "targets" / BINDER_PDB_NAME

if not target_pdb_path.exists() or not binder_pdb_path.exists():
    print(f"❌ 错误: 找不到 PDB 文件。")
    sys.exit(1)

inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
if inference_dir.exists(): shutil.rmtree(inference_dir)
inference_dir.mkdir(parents=True, exist_ok=True)

mock_design_id = f"job_{JOB_ID}_native_baseline"
mock_job_dir = inference_dir / mock_design_id
mock_job_dir.mkdir(parents=True, exist_ok=True)

# --- [2. 核心补丁一：通用参数吸收器] ---
utils_file = BASE_DIR / 'src/proteinfoundation/metrics/metric_utils.py'
if utils_file.exists():
    with open(utils_file, 'r') as f: content = f.read()
    if "def replace_seq_in_generated_pdb(" in content and "def _disabled_replace" not in content:
        shutil.copy(utils_file, str(utils_file) + ".original")
        safe_override = (
            "def replace_seq_in_generated_pdb(*args, **kwargs):\n"
            "    import shutil, os\n"
            "    for val in list(args) + list(kwargs.values()):\n"
            "        if isinstance(val, str) and val.endswith('.pdb'):\n"
            "            out_path = val.replace('.pdb', '_updated.pdb')\n"
            "            if not os.path.exists(out_path): shutil.copy(val, out_path)\n"
            "    return\n\n"
            "def _disabled_replace("
        )
        with open(utils_file, 'w') as f: f.write(content.replace("def replace_seq_in_generated_pdb(", safe_override))

# --- [2.5 核心补丁二：修复底层源码 Binder Metrics 的张冠李戴越界 Bug] ---
binder_metrics_file = BASE_DIR / 'src/proteinfoundation/metrics/binder_metrics.py'
if binder_metrics_file.exists():
    with open(binder_metrics_file, 'r') as f:
        bm_content = f.read()
    buggy_line = 'interface_seq = "".join([sequence[i] for i in interface_residues])'
    safe_line = 'interface_seq = "".join([sequence[i] for i in interface_residues if i < len(sequence)])'
    if buggy_line in bm_content:
        shutil.copy(binder_metrics_file, str(binder_metrics_file) + ".bak")
        with open(binder_metrics_file, 'w') as f:
            f.write(bm_content.replace(buggy_line, safe_line))
        print("✅ 成功注入底层修复：解决 interface_seq 索引越界崩溃 Bug。")

# --- [3. 自动拼装 PDB (A+B)] ---
target_pdb_eval_path = mock_job_dir / f"{mock_design_id}.pdb"
atom_idx = 1
with open(target_pdb_eval_path, 'w') as f_out:
    for chain, p in [('A', target_pdb_path), ('B', binder_pdb_path)]:
        with open(p, 'r') as f_in:
            for line in f_in:
                if line.startswith(("ATOM", "HETATM")):
                    f_out.write(line[:6] + f"{atom_idx:>5}" + line[11:21] + chain + line[22:])
                    atom_idx += 1
        f_out.write("TER\n")
    f_out.write("END\n")

# --- [4. 显存清理 & 执行引擎] ---
def run_PF_command(command_str):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    env = os.environ.copy()
    env.update({'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'TF_FORCE_GPU_ALLOW_GROWTH': 'true'})

    full_command = f"source env.sh && {command_str}"
    process = subprocess.Popen(full_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), env=env)
    for line in process.stdout: print(line, end='', flush=True)
    return process.wait()

base_args = (f"--config-path {BASE_DIR}/configs --config-name search_binder_local_pipeline "
             f"++job_id={JOB_ID} ++generation.task_name={TARGET_TASK} ++run_name={RUN_NAME} "
             f"++generation.dataloader.dataset.conditional_features.0.pdb_path={target_pdb_path} "
             f"++generation.dataloader.batch_size=1 ++hydra.job.chdir=False ++eval_nmodels=1")

print(f"\n🚀 启动 AlphaFold2 评估引擎 (双重优化模式)...")
ret = run_PF_command(f"python3 -m proteinfoundation.evaluate {base_args} ++run_evaluate=True ++eval_njobs=1")

if ret == 0:
    run_PF_command(f"python3 -m proteinfoundation.analyze {base_args} ++run_analyze=True")
    eval_out_dir = BASE_DIR / 'evaluation_results' / f'search_binder_local_pipeline_{TARGET_TASK}_{RUN_NAME}'
    csv_files = list(eval_out_dir.glob("binder_results_*.csv"))

    if csv_files:
        # 动态读取最新的 CSV 文件，拒绝硬编码
        df = pd.read_csv(max(csv_files, key=lambda x: x.stat().st_size))
        d = df.iloc[0]

        # --- [5.1 核心概览面板] ---
        print("\n🏆 Gold Standard Baseline (10AA Core Complex Reference):")
        header_format = "{:<8} | {:<15} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
        print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD"))
        print("-" * 85)
        print(header_format.format(
            "CORE", "GFP_11_CORE",
            f"{d.get('self_complex_min_ipAE', 0):.4f}",
            f"{d.get('self_complex_pTM', 0):.4f}",
            f"{d.get('self_complex_i_pTM', 0):.4f}",
            f"{d.get('self_complex_pLDDT', 0):.2f}",
            f"{d.get('self_binder_scRMSD_ca', 0):.2f}Å"
        ))
        print("-" * 85)

        # --- [5.2 详细指标与完整数据列] ---
        print("\n🎯 --- 天然底物基线分数详细解析 (GFP_1_10.pdb + GFP_11.pdb) ---")
        for _, row in df.iterrows():
            # 使用 .get 配合双保险，兼容不同版本的指标命名
            print(f"  > i_pAE (界面误差): {row.get('self_complex_min_ipAE', row.get('complex_i_pAE', 'N/A'))}")
            print(f"  > pTM            : {row.get('self_complex_pTM', row.get('complex_pTM', 'N/A'))}")
            print(f"  > ipTM           : {row.get('self_complex_i_pTM', row.get('complex_ipTM', 'N/A'))}")
            print(f"  > pLDDT (置信度)  : {row.get('self_complex_pLDDT', row.get('complex_pLDDT', 'N/A'))}")
            print(f"  > Binder scRMSD  : {row.get('self_binder_scRMSD_ca', row.get('binder_scRMSD_ca', 'N/A'))}")
            print("\n📊 完整数据列如下：")
            print(df.T)

        # --- [6. 动态构建绝对路径的 3D 可视化] ---
        COMPLEX_PDB_PATH = eval_out_dir / mock_design_id / "AF2" / f"{mock_design_id}_self_seq_0_model1.pdb"

        if COMPLEX_PDB_PATH.exists():
            print(f"\n   -> Rendering 3D Complex Map: Target (Gray) + Native Binder (Green)...")
            with open(COMPLEX_PDB_PATH, 'r') as f:
                pdb_data = f.read()

            viewer = py3Dmol.view(width=800, height=500)
            viewer.addModel(pdb_data, 'pdb')

            # Target 灰色半透明
            target_sel = {'chain': 'A'}
            viewer.setStyle(target_sel, {'cartoon': {'color': '#A9A9A9', 'opacity': 0.8}})
            viewer.addSurface(py3Dmol.SES, {'color': '#A9A9A9', 'opacity': 0.4}, target_sel)

            # Binder 亮绿色棒状结构
            binder_sel = {'chain': 'B'}
            viewer.setStyle(binder_sel, {
                'cartoon': {'color': 'green', 'arrows': True},
                'stick': {'radius': 0.15, 'colorscheme': 'greenCarbon'}
            })

            viewer.zoomTo()
            viewer.show()
        else:
            print(f"\n   -> ⚠️ Visualization skipped: 找不到复合体 PDB 文件，请检查路径: {COMPLEX_PDB_PATH}")

else:
    print(f"\n❌ 评估失败，退出代码: {ret}。")


🚀 启动 AlphaFold2 评估引擎 (双重优化模式)...
Complexa environment initialized for uv runtime.
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
2026-04-04 20:10:54.086 | WARNING  | proteinfoundation.evaluation.binder_eval:<module>:53 - TMOL import failed: No module named 'tmol'. TMOL metrics will return NaN.
/usr/local/lib/python3.12/dist-packages/graphein/protein/config.py:217: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator("alt_locs")
Loading Atomworks Patches
[atomworks_patches] Patched initialize_chain_info_from_category
[atomworks_patches

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Section 3 Scan Protein Groove

In [ ]:
# # Cell_11_Automated_Groove_Scanning_Engine.py
# # Requirement: Implement Morphological Bottleneck Severing via Euclidean Distance Transform (EDT). Designed to mathematically sever narrow topological channels connecting surface clefts to hollow internal cores (barrel bleed-through).

# import os, json, sys, yaml
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from scipy.spatial import KDTree
# from scipy.ndimage import binary_closing, distance_transform_edt, label

# print("🧬 Initializing Morphological Bottleneck Severing Engine (Optimized for Split-Barrels)...")

# # ---------------------------------------------------------
# # 1. Load Configurations & YAML Interface
# # ---------------------------------------------------------
# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# SETTING_FILE = BASE_DIR / 'internal_settings.json'
# YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

# if not SETTING_FILE.exists() or not YAML_PATH.exists():
#     print("❌ Error: Required configuration files not found.")
#     sys.exit(1)

# with open(SETTING_FILE, 'r') as f:
#     cfg = json.load(f)

# task_name = cfg['task_name']
# target_chains = cfg['target_chains']

# scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
# scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
# scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
# scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

# MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
# HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
# CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

# with open(YAML_PATH, 'r') as f:
#     yaml_data = yaml.safe_load(f)
# try:
#     target_info = yaml_data['target_dict_cfg'][task_name]
#     pdb_filename = target_info['target_filename'] + ".pdb"
#     SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
# except KeyError:
#     print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
#     sys.exit(1)

# RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
# RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

# KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
#             'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
#             'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
# CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# # ---------------------------------------------------------
# # 2. PDB Parsing & Grid Initialization
# # ---------------------------------------------------------
# ca_coords, res_ids, res_names = [], [], []
# all_coords = []

# with open(SOURCE_PDB_PATH, 'r') as f:
#     for line in f:
#         if line.startswith("ATOM") and line[21] == target_chains:
#             coord = [float(line[30:38]), float(line[38:46]), float(line[46:54])]
#             all_coords.append(coord)
#             if line[12:16].strip() == "CA":
#                 res_names.append(line[17:20].strip())
#                 res_ids.append(int(line[22:26].strip()))
#                 ca_coords.append(coord)

# ca_coords = np.array(ca_coords)
# res_ids = np.array(res_ids)
# res_names = np.array(res_names)
# all_coords = np.array(all_coords)
# protein_kdtree = KDTree(ca_coords)

# print("   -> Constructing 3D Voxel Cartography...")
# GRID_RES = 1.0
# PADDING = 15.0
# coords_min = all_coords.min(axis=0) - PADDING
# coords_max = all_coords.max(axis=0) + PADDING
# grid_shape = np.ceil((coords_max - coords_min) / GRID_RES).astype(int)

# protein_grid = np.zeros(grid_shape, dtype=bool)
# for pt in all_coords:
#     idx = np.round((pt - coords_min) / GRID_RES).astype(int)
#     # Expand atoms slightly to form a solid wall, sealing sub-atomic cracks
#     x, y, z = idx
#     protein_grid[max(0, x-1):x+2, max(0, y-1):y+2, max(0, z-1):z+2] = True

# # ---------------------------------------------------------
# # 3. Virtual Shrink-Wrap & Bottleneck Severing
# # ---------------------------------------------------------
# print("   -> Deploying sturdy shrink-wrap boundary to define cavities...")
# # 使用 15A 半径的巨大探针球，像一块硬纸板一样盖住巨大的裂缝，防止塌陷
# PROBE_RADIUS = 15.0
# r = int(np.ceil(PROBE_RADIUS / GRID_RES))
# zz, yy, xx = np.ogrid[-r:r+1, -r:r+1, -r:r+1]
# sphere_struct = xx**2 + yy**2 + zz**2 <= r**2

# wrapped_protein = binary_closing(protein_grid, structure=sphere_struct)

# # 提取表面保鲜膜和真实蛋白之间的夹层空腔
# cavity_mask = wrapped_protein & ~protein_grid

# print("   -> Severing narrow topological channels (EDT Bottleneck Erosion)...")
# # 计算绝对空间距离：空腔中每一个点，距离最近的蛋白原子的绝对距离
# empty_space = ~protein_grid
# edt = distance_transform_edt(empty_space) * GRID_RES

# # =========================================================
# # 物理切断 (The Cut):
# # 斩断宽度小于 5A 的隐蔽狭窄通道 (因为它们的最大 EDT 只有 2.5A)。
# # 宽阔的外部沟槽中心区 (EDT > 2.5) 与内部巨型空腔中心区 (EDT > 2.5)
# # 由此在物理空间中被彻底断开连通性。
# # =========================================================
# SEVERING_RADIUS = 2.5
# isolated_cores = cavity_mask & (edt >= SEVERING_RADIUS)

# if not np.any(isolated_cores):
#     print("🛑 Error: No core cavities survived the bottleneck severing process.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 4. Spatial Clustering of Isolated Cores
# # ---------------------------------------------------------
# print("   -> Clustering cleanly severed topological features...")
# labeled_array, num_features = label(isolated_cores)
# vertex_clusters = []

# for i in range(1, num_features + 1):
#     core_indices = np.argwhere(labeled_array == i)

#     # 体积过滤: 一个真实的残缺裂谷核心体积必须显著大于局部噪音
#     if len(core_indices) >= 40:
#         voxel_coords = core_indices * GRID_RES + coords_min
#         vertex_clusters.append(voxel_coords)

# if not vertex_clusters:
#     print("🛑 Scanning Complete: No disconnected core trenches found meeting volume limits.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 5. Restoring Contact Surfaces (Dilation to CA atoms)
# # ---------------------------------------------------------
# pocket_results = []
# categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}
# count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

# for cluster_coords in vertex_clusters:
#     # 从已经被隔断的安全核心区域，向外扩张探测 6.5A 抓取 CA 墙壁氨基酸
#     neighbors = protein_kdtree.query_ball_point(cluster_coords, r=6.5)

#     res_indices = set()
#     for nl in neighbors: res_indices.update(nl)

#     if len(res_indices) < MIN_POCKET_SIZE:
#         continue

#     cluster_res_indices = list(res_indices)
#     c_res_names = res_names[cluster_res_indices]
#     total = len(cluster_res_indices)

#     # 用抓取到的氨基酸数量来判定是浅沟还是深坑
#     topo_type = "Deep Pocket" if total > 25 else "Shallow Groove"

#     hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
#     pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
#     neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
#     charge_count = pos_count + neg_count

#     hydro_ratio = hydro_count / total
#     charge_ratio = charge_count / total

#     is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
#     is_charge = charge_ratio >= CHARGED_THRESHOLD

#     c_res_ids = res_ids[cluster_res_indices]
#     patch_str = ",".join(map(str, sorted(c_res_ids)))
#     pymol_sel = "+".join(map(str, sorted(c_res_ids)))

#     site_id = f"Site_{len(pocket_results)+1}"
#     pocket_results.append({
#         'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
#         'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
#         'Hotspot_Sequence': patch_str
#     })

#     if topo_type == "Shallow Groove" and scan_shallow_groove:
#         count_groove += 1
#         spec_id = f"Groove_{count_groove}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Grooves'].append((spec_id, cmd))

#     if topo_type == "Deep Pocket" and scan_deep_pocket:
#         count_pocket += 1
#         spec_id = f"Pocket_{count_pocket}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Pockets'].append((spec_id, cmd))

#     if is_hydro and scan_hydrophobic_patch:
#         count_hydro += 1
#         spec_id = f"Hydro_Patch_{count_hydro}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd))

#     if is_charge and scan_charged_patch:
#         count_charge += 1
#         spec_id = f"Charge_Patch_{count_charge}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Charged_Patches'].append((spec_id, cmd))

# # ---------------------------------------------------------
# # 6. Printed Output
# # ---------------------------------------------------------
# print("\n" + "="*70)
# print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
# print("="*70 + "\n")

# if categorized_findings['Grooves']:
#     print("Grooves:")
#     for t_id, cmd in categorized_findings['Grooves']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Pockets']:
#     print("Pockets:")
#     for t_id, cmd in categorized_findings['Pockets']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Hydrophobic_Patches']:
#     print("Patches (Hydrophobic):")
#     for t_id, cmd in categorized_findings['Hydrophobic_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Charged_Patches']:
#     print("Patches (Charged):")
#     for t_id, cmd in categorized_findings['Charged_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if pocket_results:
#     pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# # Purpose: Fixed a typo in variable names during the morphological clustering phase.
# # Upstream Code: Fixes bug in Cell 11 line 130.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-03 21:38 EDT.
# # Changed Lines:
# # - Line 126: Corrected `label(isolated_core_mask)` to `label(isolated_cores)`.

🧬 Initializing Morphological Bottleneck Severing Engine (Optimized for Split-Barrels)...
   -> Constructing 3D Voxel Cartography...
   -> Deploying sturdy shrink-wrap boundary to define cavities...
   -> Severing narrow topological channels (EDT Bottleneck Erosion)...
   -> Clustering cleanly severed topological features...

Summary: 1 Grooves, 0 Pockets, 0 Hydro Patches, 0 Charged Patches.

Grooves:
  Groove_1:
  select Groove_1, chain A and resi 39+40+41+73+74+200+201+202+203+204; color orange, Groove_1; show surface, Groove_1



In [ ]:
# # Cell_11_Automated_Groove_Scanning_Engine.py
# # Requirement: Paradigm shift to Opposing Ridges Void Midpoint (ORVM) algorithm. Specifically targets the user's request to exclusively highlight the two parallel ridges forming a trench. It identifies pairs of sequentially distant atoms separated by an 8.5-13.5A continuous void, meticulously isolating the "left bank" and "right bank" of missing beta-strands.

# import os, json, sys, yaml
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from scipy.spatial import KDTree

# print("🔍 Initializing Opposing Ridges Void Midpoint (ORVM) Engine...")

# # ---------------------------------------------------------
# # 1. Load Configurations & YAML Interface
# # ---------------------------------------------------------
# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# SETTING_FILE = BASE_DIR / 'internal_settings.json'
# YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

# if not SETTING_FILE.exists() or not YAML_PATH.exists():
#     print("❌ Error: Required configuration files not found.")
#     sys.exit(1)

# with open(SETTING_FILE, 'r') as f:
#     cfg = json.load(f)

# task_name = cfg['task_name']
# target_chains = cfg['target_chains']

# scan_shallow_groove = bool(cfg.get('scan_shallow_groove', True))
# scan_deep_pocket = bool(cfg.get('scan_deep_pocket', False))
# scan_hydrophobic_patch = bool(cfg.get('scan_hydrophobic_patch', True))
# scan_charged_patch = bool(cfg.get('scan_charged_patch', False))

# MIN_POCKET_SIZE = int(cfg.get('min_pocket_size', 6))
# HYDROPHOBIC_THRESHOLD = float(cfg.get('hydrophobic_threshold', 0.35))
# CHARGED_THRESHOLD = float(cfg.get('charged_threshold', 0.30))

# with open(YAML_PATH, 'r') as f:
#     yaml_data = yaml.safe_load(f)
# try:
#     target_info = yaml_data['target_dict_cfg'][task_name]
#     pdb_filename = target_info['target_filename'] + ".pdb"
#     SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / target_info['source'] / pdb_filename
# except KeyError:
#     print(f"❌ Error: Target '{task_name}' not properly defined in YAML.")
#     sys.exit(1)

# RESULTS_CSV = BASE_DIR / 'screening_results' / task_name / f"{task_name}_automated_scans.csv"
# RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

# KD_SCALE = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
#             'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
#             'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
# CHARGE_SCALE = {'ARG': '+', 'LYS': '+', 'HIS': '+', 'ASP': '-', 'GLU': '-'}

# # ---------------------------------------------------------
# # 2. PDB Parsing & Coordinate Extraction
# # ---------------------------------------------------------
# ca_coords, res_ids, res_names = [], [], []

# with open(SOURCE_PDB_PATH, 'r') as f:
#     for line in f:
#         if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
#             res_names.append(line[17:20].strip())
#             res_ids.append(int(line[22:26].strip()))
#             ca_coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

# ca_coords = np.array(ca_coords)
# res_ids = np.array(res_ids)
# res_names = np.array(res_names)
# protein_kdtree = KDTree(ca_coords)

# # ---------------------------------------------------------
# # 3. Opposing Ridges Void Midpoint (ORVM) Algorithm
# # ---------------------------------------------------------
# print("   -> Scanning for opposing structural ridges separated by a continuous void...")

# # 提取距离在 14.0A 以内的所有 CA 原子对
# pairs = protein_kdtree.query_pairs(r=14.0)
# valid_pairs = []
# midpoints = []

# for i, j in pairs:
#     # a. 物理跨沟距离：缺失一条 Beta 链的鸿沟，两岸距离必定在 8.5A 到 13.5A 之间
#     dist = np.linalg.norm(ca_coords[i] - ca_coords[j])
#     if dist < 8.5 or dist > 13.5:
#         continue

#     # b. 序列断连过滤：确保是两条空间并排的不同链，而不是沿着骨架连续的链
#     if np.abs(res_ids[i] - res_ids[j]) < 15:
#         continue

#     # c. 真空靶心生成：计算两人之间的空间几何中心
#     M = (ca_coords[i] + ca_coords[j]) / 2.0

#     # d. 中点净空判定：中点必须是真空的（距离任何 CA 骨架原子至少 3.8A 以上）
#     nearest_dist, _ = protein_kdtree.query(M, k=1)
#     if nearest_dist < 3.8:
#         continue

#     # e. 内外空间绝杀 (Surface vs Core Filter)：
#     # 真正的表面沟槽，中点只被底部和两侧包围 (原子数 10-32)。
#     # 如果掉入发色团中空内腔，会被四面八方的整个桶包围 (原子数 > 35)。
#     neighbors_count = len(protein_kdtree.query_ball_point(M, r=12.0))
#     if neighbors_count < 10 or neighbors_count > 32:
#         continue

#     valid_pairs.append((i, j))
#     midpoints.append(M)

# midpoints = np.array(midpoints)

# if len(midpoints) == 0:
#     print("🛑 Scanning Complete: No opposing ridges across a valid topological trench were found.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 4. Clustering Validated Midpoints into Continuous Trenches
# # ---------------------------------------------------------
# print("   -> Stitching cross-trench voids to isolate distinct binding grooves...")
# midpoint_kdtree = KDTree(midpoints)

# # 如果多个真空靶心相距极近(4.5A)，说明它们属于同一条连续的沟槽
# mp_pairs = midpoint_kdtree.query_pairs(r=4.5)

# adj_list = {i: set() for i in range(len(midpoints))}
# for i, j in mp_pairs:
#     adj_list[i].add(j); adj_list[j].add(i)

# visited = set()
# final_ridge_clusters = []

# for i in range(len(midpoints)):
#     if i not in visited:
#         queue = [i]
#         component = []
#         while queue:
#             node = queue.pop(0)
#             if node not in visited:
#                 visited.add(node)
#                 component.append(node)
#                 queue.extend(list(adj_list[node] - visited))

#         # 一个真实的生物学裂谷，至少会生成 8 个以上的有效真空靶心对
#         if len(component) >= 8:
#             # 提取贡献了这条沟槽的所有左岸和右岸的原子索引
#             ridge_indices = set()
#             for idx in component:
#                 u, v = valid_pairs[idx]
#                 ridge_indices.add(u)
#                 ridge_indices.add(v)
#             final_ridge_clusters.append(list(ridge_indices))

# if not final_ridge_clusters:
#     print("🛑 Scanning Complete: Detected opposing ridges were too fragmented to form a continuous groove.")
#     sys.exit(0)

# # ---------------------------------------------------------
# # 5. Categorization & Annotation
# # ---------------------------------------------------------
# pocket_results = []
# categorized_findings = {'Grooves': [], 'Pockets': [], 'Hydrophobic_Patches': [], 'Charged_Patches': []}

# count_groove = 0; count_pocket = 0; count_hydro = 0; count_charge = 0

# for cluster in final_ridge_clusters:
#     c_res_names = res_names[cluster]
#     total = len(cluster)

#     # 这种配对算法抓取的纯粹是两侧高耸的山脊
#     topo_type = "Deep Pocket" if total > 35 else "Shallow Groove"

#     hydro_count = sum(1 for name in c_res_names if KD_SCALE.get(name, 0) > 0)
#     pos_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '+')
#     neg_count = sum(1 for name in c_res_names if CHARGE_SCALE.get(name) == '-')
#     charge_count = pos_count + neg_count

#     hydro_ratio = hydro_count / total
#     charge_ratio = charge_count / total

#     is_hydro = hydro_ratio >= HYDROPHOBIC_THRESHOLD
#     is_charge = charge_ratio >= CHARGED_THRESHOLD

#     c_res_ids = res_ids[cluster]
#     patch_str = ",".join(map(str, sorted(c_res_ids)))
#     pymol_sel = "+".join(map(str, sorted(c_res_ids)))

#     site_id = f"Site_{len(pocket_results)+1}"
#     pocket_results.append({
#         'Site_ID': site_id, 'Topology': topo_type, 'Residues': total,
#         'Hydrophobic_Ratio': round(hydro_ratio, 3), 'Charged_Ratio': round(charge_ratio, 3),
#         'Hotspot_Sequence': patch_str
#     })

#     if topo_type == "Shallow Groove" and scan_shallow_groove:
#         count_groove += 1
#         spec_id = f"Groove_{count_groove}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color orange, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Grooves'].append((spec_id, cmd))

#     if topo_type == "Deep Pocket" and scan_deep_pocket:
#         count_pocket += 1
#         spec_id = f"Pocket_{count_pocket}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color magenta, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Pockets'].append((spec_id, cmd))

#     if is_hydro and scan_hydrophobic_patch:
#         count_hydro += 1
#         spec_id = f"Hydro_Patch_{count_hydro}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color yellow, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Hydrophobic_Patches'].append((spec_id, cmd))

#     if is_charge and scan_charged_patch:
#         count_charge += 1
#         spec_id = f"Charge_Patch_{count_charge}"
#         cmd = f"select {spec_id}, chain {target_chains} and resi {pymol_sel}; color cyan, {spec_id}; show surface, {spec_id}"
#         categorized_findings['Charged_Patches'].append((spec_id, cmd))

# # ---------------------------------------------------------
# # 6. Printed Output
# # ---------------------------------------------------------
# print("\n" + "="*70)
# print(f"Summary: {count_groove} Grooves, {count_pocket} Pockets, {count_hydro} Hydro Patches, {count_charge} Charged Patches.")
# print("="*70 + "\n")

# if categorized_findings['Grooves']:
#     print("Grooves:")
#     for t_id, cmd in categorized_findings['Grooves']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Pockets']:
#     print("Pockets:")
#     for t_id, cmd in categorized_findings['Pockets']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Hydrophobic_Patches']:
#     print("Patches (Hydrophobic):")
#     for t_id, cmd in categorized_findings['Hydrophobic_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if categorized_findings['Charged_Patches']:
#     print("Patches (Charged):")
#     for t_id, cmd in categorized_findings['Charged_Patches']:
#         print(f"  {t_id}:\n  {cmd}\n")

# if pocket_results:
#     pd.DataFrame(pocket_results).to_csv(RESULTS_CSV, index=False)

# # ==============================================================================
# # Update Log:
# # Purpose: Deployed the Opposing Ridges Void Midpoint (ORVM) algorithm specifically designed to fulfill the user's intent to highlight *only* the two parallel ridges flanking a trench. It calculates geometric midpoints between sequentially distant atoms (gap 8.5-13.5A). It strictly guarantees void clearance (no CA within 3.8A) and rejects central barrel voids by limiting neighborhood counts (10-32 atoms within 12A). By extracting only the CA pairs generating these clustered void midpoints, it provides an exquisite isolation of trench lips with zero bleed to the floor or surrounding convex topology.
# # Upstream Code: Complete algorithmic refactor of Section 3-4.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-03 22:15 EDT.
# # Changed Lines:
# # - Lines 85-115: Implemented Void Midpoint generation and verification logic.
# # - Lines 103: `nearest_dist < 3.8` enforces perfect physical emptiness between the two ridges.
# # - Lines 110: `neighbors_count < 10 or neighbors_count > 32` prevents central GFP barrel hollows from registering as surface grooves.
# # - Lines 142-146: Extracted only the original `(u, v)` pair atom indices that formed the valid trench void, directly producing the two distinct mountain ridges.
# # ==============================================================================

# Section 6: High-Throughput Autonomous Screening

In [ ]:

# Cell_9b_Autonomous_Generation_Loop.py
# Requirement: Execute an automated loop for high-throughput generation and evaluation.
# Implement a hierarchical directory structure (Master Session -> Cycle Subfolders) to isolate PDB references and prevent I/O race conditions during continuous loops. Integrate robust CSV parsing and leaderboard reporting.

import os, json, sys, time, subprocess, csv, datetime, shutil
from pathlib import Path
from Bio.SeqUtils.ProtParam import ProteinAnalysis
# ---------------------------------------------------------
# 1. Load Persistent Configuration
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please navigate to Section 0 and configure parameters.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_threshold = float(cfg['target_threshold'])
target_success_count = int(cfg['target_success_count'])
designs_per_loop = int(cfg['designs_per_loop'])
max_iterations = int(cfg['max_iterations'])
batch_size = int(cfg['batch_size'])

print(f"🤖 Auto-Pilot Engine Activated for Target: [{task_name}]")
print(f"🎯 Objective Strategy: Capture {target_success_count} qualified hits surpassing threshold {target_threshold}.")
print(f"🔄 Loop Matrix: Designing {designs_per_loop} structures per cycle (Maximum threshold: {max_iterations} cycles).")

# ---------------------------------------------------------
# 2. Tracking Variables, Logging & Master Hierarchy Setup
# ---------------------------------------------------------
current_iteration = 0
ALL_DESIGNS_POOL = []
GLOBAL_HITS = []

SCREENING_RESULTS_DIR = BASE_DIR / 'screening_results' / task_name
SCREENING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
master_log_path = SCREENING_RESULTS_DIR / "autopilot_master_log.csv"
MASTER_LOG_FILENAME = master_log_path.name

# Define the absolute source of the initial processed PDB
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Required source PDB structure not found at {SOURCE_PDB_PATH}. Run Section 2 first.")
    sys.exit(1)

# Establish the Master Session Directory for this specific Auto-Pilot run
SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"AutoPilot_{task_name}_{SESSION_TIMESTAMP}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Master Session Directory established at: {SESSION_MASTER_DIR.relative_to(BASE_DIR)}")

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m}m {s}s"

env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

# ---------------------------------------------------------
# ---------------------------------------------------------
# 3. High-Throughput Engine Execution Loop
# ---------------------------------------------------------
# ========== Code Modification 1: Initialize global start time before the while loop ==========
global_start_time = time.time()

while current_iteration < max_iterations and len(GLOBAL_HITS) < target_success_count:
    current_iteration += 1
    loop_start_time = time.time()
    batch_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Generate isolated unique run nomenclature for the present cycle
    current_run_name = f"{SESSION_NAME}_Cycle{current_iteration}"

    # Create the isolated Sub-Directory for this specific cycle within the Master Directory
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / current_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    # Mirror the PDB to the isolated directory to prevent concurrent access conflicts
    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 Auto-Pilot Cycle Iteration [{current_iteration}/{max_iterations}] Commencing...")
    print(f"📂 Cycle Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    # Native Complexa Command Parameterization - Target isolated sandbox path
    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={current_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={designs_per_loop} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=True ++run_analyze=False"
    )

    # Process bridging for generation and evaluation
    # --- Dynamic Monitoring Module Start ---
    iter_start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n🕒 Cycle {current_iteration} Start Time: {iter_start_str}")

    if current_iteration > 1:
        avg_cycle_time = (time.time() - global_start_time) / (current_iteration - 1)
        remaining_cycles = max_iterations - current_iteration + 1
        eta_seconds = avg_cycle_time * remaining_cycles
        eta_str = (datetime.datetime.now() + datetime.timedelta(seconds=eta_seconds)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"⏳ Estimated Time of Completion (ETA) for max cycles: {eta_str}")
    else:
        print(f"⏳ Estimated Time of Completion (ETA): Calculating after first cycle...")

    process = subprocess.Popen(
        f"source env.sh && {cmd_str}",
        env=env,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        # Maintain native underlying log output
        print(line, end='', flush=True)

        # Dynamically parse underlying logs to output monitoring indicators
        lower_line = line.lower()
        if "generation" in lower_line and "start" in lower_line:
            print(f"\n>>> [Monitor] Phase transition: Generation Sequence Initiated...")
        elif "evaluate" in lower_line and "start" in lower_line:
            print(f"\n>>> [Monitor] Phase transition: Evaluation Sequence Initiated...")
        elif "batch" in lower_line and "/" in lower_line:
            print(f">>> [Monitor] Internal Progress: {line.strip()}")

    process.wait()
    ret_code = process.returncode
    # --- Dynamic Monitoring Module End ---


    # ---------------------------------------------------------
    # 4. Report Parsing & Dynamic Leaderboard Allocation
    # ---------------------------------------------------------
    if ret_code == 0:
        # Dynamically resolve generated infrastructure based on the new cycle name
        inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{current_run_name}'
        valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

        if valid_csvs:
            target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
            with open(target_csv, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)

                for i, row in enumerate(reader):
                    try:
                        ipsae_val = float(row.get('af2folding_min_ipsae', -999.0))
                        ptm_val = float(row.get('af2folding_ptm_log', -999.0))
                        iptm_val = float(row.get('af2folding_i_ptm_log', -999.0))
                        plddt_val = float(row.get('af2folding_plddt', -999.0))
                        rmsd_val = float(row.get('af2folding_rmsd', -999.0))
                        total_reward = float(row.get('total_reward', -999.0))

                        d_id = row.get('design_id', '').strip() or f"Design_{row.get('pdb_index', i)}"

                        design_entry = {
                            'id': d_id, 'score': ipsae_val, 'ptm': ptm_val, 'iptm': iptm_val,
                            'plddt': plddt_val, 'rmsd': rmsd_val, 'total_reward': total_reward,
                            'batch_num': current_iteration, 'idx_in_batch': i + 1, 'dir': inference_dir, 'loop': current_iteration
                        }
                        ALL_DESIGNS_POOL.append(design_entry)

                        if ipsae_val >= target_threshold:
                            if not any(h['id'] == d_id and h['dir'] == inference_dir for h in GLOBAL_HITS):
                                GLOBAL_HITS.append(design_entry)
                    except Exception as e:
                        pass

            # Persistent appends to the central Master Log
            file_exists = master_log_path.exists()
            with open(master_log_path, 'a', encoding='utf-8', newline='') as f_log:
                fieldnames = ['session_loop', 'timestamp', 'design_id', 'min_ipSAE', 'pTM', 'iPTM', 'pLDDT', 'RMSD', 'total_reward', 'source_path']
                writer = csv.DictWriter(f_log, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()

                new_entries = [d for d in ALL_DESIGNS_POOL if d['loop'] == current_iteration]
                for entry in new_entries:
                    writer.writerow({
                        'session_loop': entry['loop'], 'timestamp': batch_timestamp, 'design_id': entry['id'],
                        'min_ipSAE': f"{entry['score']:.4f}", 'pTM': f"{entry['ptm']:.4f}",
                        'iPTM': f"{entry['iptm']:.4f}", 'pLDDT': f"{entry['plddt']:.2f}",
                        'RMSD': f"{entry['rmsd']:.2f}", 'total_reward': f"{entry['total_reward']:.4f}",
                        'source_path': str(entry['dir'].relative_to(BASE_DIR))
                    })
            print(f"📝 Transferred {len(new_entries)} evaluated construct metrics directly to {MASTER_LOG_FILENAME}")
        else:
            print("⚠️ Warning: Pipeline succeeded, but no valid CSV output files were found.")
    else:
        print(f"⚠️ Alert: Generative pipeline exited with non-zero status code {ret_code}. Skipping metric ingestion for Cycle {current_iteration}.")

    # ---------------------------------------------------------
    # Cell_9b_Autonomous_Generation_Loop.py
# Requirement: Update the generation loop to calculate exact total designs based on math.ceil(designs/batch), inject a dynamic random seed explicitly into the command string, enable run_analyze=True, and merge with the dynamic stdout monitoring module.

# ---------------------------------------------------------
# 3. High-Throughput Engine Execution Loop
# ---------------------------------------------------------
import math
import random

global_start_time = time.time()

while current_iteration < max_iterations and len(GLOBAL_HITS) < target_success_count:
    current_iteration += 1
    loop_start_time = time.time()
    batch_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Generate isolated unique run nomenclature for the present cycle
    current_run_name = f"{SESSION_NAME}_Cycle{current_iteration}"

    # Create the isolated Sub-Directory for this specific cycle within the Master Directory
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / current_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    # Mirror the PDB to the isolated directory to prevent concurrent access conflicts
    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 Auto-Pilot Cycle Iteration [{current_iteration}/{max_iterations}] Commencing...")
    print(f"📂 Cycle Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    # Calculate exact total designs to prevent batch remainder issues
    num_batches = math.ceil(designs_per_loop / batch_size)
    total_designs = num_batches * batch_size
    current_seed = random.randint(1, 1000000)

    # Native Complexa Command Parameterization - Target isolated sandbox path
    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={current_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={total_designs} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=True ++run_analyze=True ++seed={current_seed}"
    )

    print(f"🔥 Transmitting execution command to the underlying engine (Seed: {current_seed})...")

    # --- Dynamic Monitoring Module Start ---
    iter_start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n🕒 Cycle {current_iteration} Start Time: {iter_start_str}")

    if current_iteration > 1:
        avg_cycle_time = (time.time() - global_start_time) / (current_iteration - 1)
        remaining_cycles = max_iterations - current_iteration + 1
        eta_seconds = avg_cycle_time * remaining_cycles
        eta_str = (datetime.datetime.now() + datetime.timedelta(seconds=eta_seconds)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"⏳ Estimated Time of Completion (ETA) for max cycles: {eta_str}")
    else:
        print(f"⏳ Estimated Time of Completion (ETA): Calculating after first cycle...")

    process = subprocess.Popen(
        f"source env.sh && {cmd_str}",
        env=env,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        # Maintain native underlying log output
        print(line, end='', flush=True)

        # Dynamically parse underlying logs to output monitoring indicators
        lower_line = line.lower()
        if "generation" in lower_line and "start" in lower_line:
            print(f"\n>>> [Monitor] Phase transition: Generation Sequence Initiated...")
        elif "evaluate" in lower_line and "start" in lower_line:
            print(f"\n>>> [Monitor] Phase transition: Evaluation Sequence Initiated...")
        elif "batch" in lower_line and "/" in lower_line:
            print(f">>> [Monitor] Internal Progress: {line.strip()}")

    process.wait()
    ret_code = process.returncode
    # --- Dynamic Monitoring Module End ---

    if ret_code != 0:
        print(f"❌ Critical Warning: Underlying generation engine crashed (Error Code {ret_code})!")
        time.sleep(3)
        continue

    # ---------------------------------------------------------
    # 4. Report Parsing & Dynamic Leaderboard Allocation
    # ---------------------------------------------------------
    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{current_run_name}'
    valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

    if valid_csvs:
        target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
        with open(target_csv, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)

            for i, row in enumerate(reader):
                try:
                    ipsae_val = float(row.get('af2folding_min_ipsae', -999.0))
                    ptm_val = float(row.get('af2folding_ptm_log', -999.0))
                    iptm_val = float(row.get('af2folding_i_ptm_log', -999.0))
                    plddt_val = float(row.get('af2folding_plddt', -999.0))
                    rmsd_val = float(row.get('af2folding_rmsd', -999.0))
                    total_reward = float(row.get('total_reward', -999.0))

                    d_id = row.get('design_id', '').strip() or f"Design_{row.get('pdb_index', i)}"

                    design_entry = {
                        'id': d_id, 'score': ipsae_val, 'ptm': ptm_val, 'iptm': iptm_val,
                        'plddt': plddt_val, 'rmsd': rmsd_val, 'total_reward': total_reward,
                        'batch_num': current_iteration, 'idx_in_batch': i + 1, 'dir': inference_dir, 'loop': current_iteration
                    }
                    ALL_DESIGNS_POOL.append(design_entry)

                    if ipsae_val >= target_threshold:
                        if not any(h['id'] == d_id and h['dir'] == inference_dir for h in GLOBAL_HITS):
                            GLOBAL_HITS.append(design_entry)
                except Exception as e:
                    pass

        # Persistent appends to the central Master Log
        file_exists = master_log_path.exists()
        with open(master_log_path, 'a', encoding='utf-8', newline='') as f_log:
            fieldnames = ['session_loop', 'timestamp', 'design_id', 'min_ipSAE', 'pTM', 'iPTM', 'pLDDT', 'RMSD', 'total_reward', 'source_path']
            writer = csv.DictWriter(f_log, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()

            new_entries = [d for d in ALL_DESIGNS_POOL if d['loop'] == current_iteration]
            for entry in new_entries:
                writer.writerow({
                    'session_loop': entry['loop'], 'timestamp': batch_timestamp, 'design_id': entry['id'],
                    'min_ipSAE': f"{entry['score']:.4f}", 'pTM': f"{entry['ptm']:.4f}",
                    'iPTM': f"{entry['iptm']:.4f}", 'pLDDT': f"{entry['plddt']:.2f}",
                    'RMSD': f"{entry['rmsd']:.2f}", 'total_reward': f"{entry['total_reward']:.4f}",
                    'source_path': str(entry['dir'].relative_to(BASE_DIR))
                })
        print(f"📝 {len(new_entries)} designs seamlessly appended to {MASTER_LOG_FILENAME}")
    else:
        print("⚠️ Warning: Pipeline succeeded, but no valid CSV output files were found.")

    # ---------------------------------------------------------
    # 5. Cycle Summary & Execution Metrics Display
    # ---------------------------------------------------------
    cycle_duration = format_time(time.time() - loop_start_time)
    print(f"\n⏱️ Cycle {current_iteration} Operational Duration: {cycle_duration}")
    print(f"📈 Verified Candidates Captured (ipSAE >= {target_threshold}): {len(GLOBAL_HITS)} / {target_success_count}")

    ALL_DESIGNS_POOL.sort(key=lambda x: x['score'], reverse=True)

    print("\n🏆 Live Top 10 Design Roster (Global Index prioritized by min_ipSAE):")
    header_format = "{:<5} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<6} | {:<6}"
    print(header_format.format("Rank", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD", "Reward", "Batch", "Index"))
    print("-" * 105)

    if not ALL_DESIGNS_POOL:
        print("  (No valid designs generated yet. Waiting for successful cycles...)")
    else:
        for rank, d in enumerate(ALL_DESIGNS_POOL[:10], 1):
            print(header_format.format(
                rank, d['id'][:12], f"{d['score']:.4f}", f"{d['ptm']:.4f}", f"{d['iptm']:.4f}",
                f"{d['plddt']:.2f}", f"{d['rmsd']:.2f}Å", f"{d['total_reward']:.4f}", str(d['batch_num']), str(d['idx_in_batch'])
            ))
    print("-" * 105)

    # VRAM Garbage Collection Hook
    import gc
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    if len(GLOBAL_HITS) >= target_success_count:
        print(f"\n🎉 Threshold Attained! Successfully assimilated {target_success_count} rigorously vetted structural models meeting standard criteria.")
        break

    if current_iteration < max_iterations:
        print("🧹 Buffer memory cleared. Entering brief thermal cool-down before subsequent cycle initiation...")
        time.sleep(5)

# ---------------------------------------------------------
# 6. Final Execution Post-Mortem Output
# ---------------------------------------------------------
print(f"\n🛑 Auto-Pilot execution state terminated.")
if len(GLOBAL_HITS) >= target_success_count:
    print(f"🏆 OBJECTIVE COMPLETED: Hit quota satisfied yielding {len(GLOBAL_HITS)} high-affinity models.")
else:
    print(f"⚠️ MAXIMUM ITERATIONS TRIGGERED: Algorithm halted with {len(GLOBAL_HITS)} viable candidates isolated over {current_iteration} execution cycles.")

print(f"📂 Persistent Master Log registry accessible via: {master_log_path}")

# ==============================================================================
# Update Log:
# Purpose: Merge the math.ceil batch logic, random seed injection, and run_analyze toggle from the legacy snippet with the current live stdout monitoring loop.
# Upstream Code: Extends Cell_9b_Autonomous_Generation_Loop.py.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 10:06 EDT
# Changed Lines:
# * Lines 35-37: Added math.ceil logic and total_designs calculation.
# * Line 44: Appended ++run_analyze=True ++seed={current_seed} to cmd_str.
# * Lines 92-95: Updated error handling loop to catch non-zero return codes immediately.
# * Lines 142: Integrated custom print statement for seamless appended designs.
# ==============================================================================

🤖 Auto-Pilot Engine Activated for Target: [GFP]
🎯 Objective Strategy: Capture 10 qualified hits surpassing threshold 0.8.
🔄 Loop Matrix: Designing 18 structures per cycle (Maximum threshold: 50 cycles).
📂 Master Session Directory established at: assets/target_data/AutoPilot_GFP_Y2026_M04_D04_H11_M59_S02

🔄 Auto-Pilot Cycle Iteration [1/50] Commencing...
📂 Cycle Sandbox: AutoPilot_GFP_Y2026_M04_D04_H11_M59_S02_Cycle1
📝 Transferred 8 evaluated construct metrics directly to autopilot_master_log.csv

⏱️ Cycle 1 Operational Duration: 0h 25m 9s
📈 Verified Candidates Captured (ipSAE >= 0.8): 0 / 10

🏆 Live Top 10 Design Roster (Global Index prioritized by min_ipSAE):
Rank  | Design_ID    | ipSAE    | pTM      | iPTM     | pLDDT    | RMSD     | Reward   | Batch  | Index 
---------------------------------------------------------------------------------------------------------
1     | Design_6     | 0.5789   | 0.9153   | 0.8969   | 0.09     | 0.62Å    | -0.1351  | 1      | 7     
2     | Design_

KeyboardInterrupt: 

In [ ]:

# Cell_13_Top10_Dual_Color_Viewer.py
# Requirement: Extract and render the top 10 generated designs from the Auto-Pilot master log.
# Utilizes structural subtraction against the template (fixed.pdb) to precisely color only the binder/de novo peptide.

import os, json, csv
from pathlib import Path
import py3Dmol
from IPython.display import display, HTML

# 1. Configuration and Path Setup
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("Error: Global configuration file not found.")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not master_log_path.exists() or not FIXED_PDB_PATH.exists():
    print("Error: Master log or template fixed.pdb not found. Pipeline cannot proceed.")
    import sys; sys.exit(1)

# 2. Structural Subtraction Mapping (Extract baseline set)
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

template_sel_list = []
for chain, resis in template_chains_resis.items():
    template_sel_list.append({'chain': chain, 'resi': list(resis)})

target_selection = {'or': template_sel_list}
binder_selection = {'not': target_selection}

display(HTML("<h2>--- Top 10 Generated Designs ---</h2>"))

# 3. Parse Log and Sort Top 10
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['min_ipSAE'] = float(row['min_ipSAE'])
            data.append(row)
        except ValueError:
            pass

data.sort(key=lambda x: x['min_ipSAE'], reverse=True)
top_10 = data[:10]

if not top_10:
    print("Error: No valid entries found in the master log.")
    import sys; sys.exit(1)

# 4. Render Generated Visualizations
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['min_ipSAE']

    target_pdb = None
    if source_dir.exists():
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f})</h3>"))

        with open(target_pdb, 'r') as f:
            pdb_data = f.read()

        viewer = py3Dmol.view(width=800, height=400)
        viewer.addModel(pdb_data, 'pdb')

        # Native Template (Target) - Light Gray
        viewer.setStyle(target_selection, {'cartoon': {'color': '#D3D3D3', 'opacity': 0.8}})

        # De Novo Generated Peptide (Difference Set) - Bright Cyan
        viewer.setStyle(binder_selection, {'cartoon': {'color': '#00FFFF', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'opacity': 0.5, 'color': '#00FFFF'}, binder_selection)

        viewer.zoomTo()
        viewer.show()
    else:
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f}) - PDB File Not Found</h3>"))

In [ ]:

# Cell_13_Top10_Dual_Color_Viewer.py
# Requirement: Extract and render the top 10 generated designs from the Auto-Pilot master log.
# Utilizes structural subtraction against the template (fixed.pdb) to precisely color only the binder/de novo peptide.

import os, json, csv
from pathlib import Path
import py3Dmol
from IPython.display import display, HTML

# 1. Configuration and Path Setup
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("Error: Global configuration file not found.")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not master_log_path.exists() or not FIXED_PDB_PATH.exists():
    print("Error: Master log or template fixed.pdb not found. Pipeline cannot proceed.")
    import sys; sys.exit(1)

# 2. Structural Subtraction Mapping (Extract baseline set)
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

template_sel_list = []
for chain, resis in template_chains_resis.items():
    template_sel_list.append({'chain': chain, 'resi': list(resis)})

target_selection = {'or': template_sel_list}
binder_selection = {'not': target_selection}

display(HTML("<h2>--- Top 10 Generated Designs ---</h2>"))

# 3. Parse Log and Sort Top 10
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['min_ipSAE'] = float(row['min_ipSAE'])
            data.append(row)
        except ValueError:
            pass

data.sort(key=lambda x: x['min_ipSAE'], reverse=True)
top_10 = data[:10]

if not top_10:
    print("Error: No valid entries found in the master log.")
    import sys; sys.exit(1)

# 4. Render Generated Visualizations
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['min_ipSAE']

    target_pdb = None
    if source_dir.exists():
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f})</h3>"))

        with open(target_pdb, 'r') as f:
            pdb_data = f.read()

        viewer = py3Dmol.view(width=800, height=400)
        viewer.addModel(pdb_data, 'pdb')

        # Native Template (Target) - Light Gray
        viewer.setStyle(target_selection, {'cartoon': {'color': '#D3D3D3', 'opacity': 0.8}})

        # De Novo Generated Peptide (Difference Set) - Bright Cyan
        viewer.setStyle(binder_selection, {'cartoon': {'color': '#00FFFF', 'opacity': 1.0}})
        viewer.addSurface(py3Dmol.SES, {'opacity': 0.5, 'color': '#00FFFF'}, binder_selection)

        viewer.zoomTo()
        viewer.show()
    else:
        display(HTML(f"<h3>Rank {rank}: {d_id} (ipSAE: {score:.4f}) - PDB File Not Found</h3>"))

In [ ]:

# =================================================================
# 💧 Cell_14_Top10_Solubility_Analyzer.py
# Requirement: Extract the Top 10 generated designs, dynamically identify
# the binder sequence using structural subtraction, and evaluate solubility.
# =================================================================

import os, json, csv
from pathlib import Path

# 确保安装并导入 Biopython
try:
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
except ImportError:
    import subprocess
    print("安装 Biopython 中...")
    subprocess.check_call(["pip", "install", "biopython"])
    from Bio.SeqUtils.ProtParam import ProteinAnalysis

# 1. 基础配置与路径 (与 Cell 13 完全一致接口)
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ 错误: 找不到全局配置文件。")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'
FIXED_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not master_log_path.exists() or not FIXED_PDB_PATH.exists():
    print("❌ 错误: 找不到日志文件或固定模板文件。")
    import sys; sys.exit(1)

# 2. 建立 Target 排除字典 (用于提取生成的肽段)
template_chains_resis = {}
with open(FIXED_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            chain = line[21]
            resi = line[22:26].strip()
            if chain not in template_chains_resis:
                template_chains_resis[chain] = set()
            template_chains_resis[chain].add(resi)

# 氨基酸三字母转单字母字典
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
         'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
         'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
         'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

# 3. 读取 Log 并获取 Top 10
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['min_ipSAE'] = float(row['min_ipSAE'])
            data.append(row)
        except ValueError:
            pass

data.sort(key=lambda x: x['min_ipSAE'], reverse=True)
top_10 = data[:10]

if not top_10:
    print("❌ 错误: Log 中没有有效数据。")
    import sys; sys.exit(1)

print("\n🚀 --- Top 10 候选产物溶解度与理化性质评估 ---")

# 4. 遍历前 10 名并进行序列提取与分析
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['min_ipSAE']

    # 找寻对应的 PDB 文件
    target_pdb = None
    if source_dir.exists():
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        # 提取 Binder 序列
        binder_seq = ""
        with open(target_pdb, 'r') as f:
            for line in f:
                if line.startswith("ATOM") and line[12:16].strip() == "CA":
                    chain = line[21]
                    resi = line[22:26].strip()
                    resn = line[17:20].strip()
                    # 如果不在 target 字典里，说明是生成的 binder
                    if chain not in template_chains_resis or resi not in template_chains_resis[chain]:
                        binder_seq += d3to1.get(resn, 'X')

        # 过滤掉无法识别的残基（防止报错）
        binder_seq = binder_seq.replace('X', '')

        if not binder_seq:
            print(f"\n[{rank}] {d_id} | ⚠️ 无法从 PDB 提取到 Binder 序列。")
            continue

        # 核心：使用 Biopython 分析
        analysis = ProteinAnalysis(binder_seq)
        gravy_score = analysis.gravy()
        pi_value = analysis.isoelectric_point()
        charge_at_7_4 = analysis.charge_at_pH(7.4)

        # 打印面板
        print(f"\n🏅 Rank {rank} | Design: {d_id} | ipSAE: {score:.4f}")
        print(f"   🧬 提取序列 : {binder_seq}")

        # GRAVY 评估
        if gravy_score < 0:
            print(f"   💧 GRAVY 得分 : {gravy_score:.3f} (亲水，水溶性通常较好 ✅)")
        else:
            print(f"   🪨 GRAVY 得分 : {gravy_score:.3f} (疏水，有聚集/沉淀风险 ⚠️)")

        # pI 评估
        if 6.5 <= pi_value <= 8.5:
            print(f"   ⚡ 等电点(pI) : {pi_value:.2f} (⚠️ 警告：接近生理 pH 7.4，可能导致沉淀！)")
        else:
            print(f"   ⚡ 等电点(pI) : {pi_value:.2f} (远离中性 pH，有利于溶解 ✅)")

        print(f"   🔋 生理净电荷 : {charge_at_7_4:.2f} (绝对值越大，静电排斥越强，越不易聚集)")

    else:
        print(f"\n🏅 Rank {rank} | {d_id} | ❌ 找不到 PDB 文件，跳过分析。")

print("\n" + "="*60)
print("📌 分析指南:")
print("- GRAVY < 0 且 pI 远离 7.4 的多肽，在实际湿实验中最容易溶解。")
print("- 如果某序列结合分数很高，但疏水性(GRAVY)很强，建议后续对非结合面增加亲水突变。")

# Section 5: Pipeline Execution: Generative Inference

In [ ]:
# Cell_5b_Standard_Pipeline_Execution.py
# Requirement: Execute the standard structural generation and evaluation pipeline.
# Establish an isolated working directory for each run and duplicate the target PDB into it to prevent file access conflicts during repeated executions.

import os, json, sys, subprocess, datetime, shutil
from pathlib import Path

# ---------------------------------------------------------
# 1. Load Persistent Configuration (The "Memory Card")
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run Section 0 first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
num_designs = cfg['num_designs']
gen_batch_size = cfg['gen_batch_size']
run_filter = cfg['run_filter']
run_evaluate = cfg['run_evaluate']
run_analyze = cfg['run_analyze']

print(f"🚀 Initiating Generation Pipeline for Target: [{task_name}]")
print(f"📊 Designs: {num_designs} | Batch Size: {gen_batch_size}")

# ---------------------------------------------------------
# 2. Path Setup & Run Identification
# ---------------------------------------------------------
RUN_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
CURRENT_RUN_NAME = f"Run_{task_name}_{RUN_TIMESTAMP}"

# Define the source PDB generated from Section 2
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

# Establish an isolated working directory for the current run
RUN_ISOLATED_DIR = BASE_DIR / 'assets' / 'target_data' / CURRENT_RUN_NAME
RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

# Define the isolated PDB path and execute the data transfer
ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"

if SOURCE_PDB_PATH.exists():
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)
else:
    print(f"❌ Error: Source PDB not found at {SOURCE_PDB_PATH}. Pipeline cannot proceed.")
    sys.exit(1)

# Save execution manifest for downstream Analyzer
RECORD_FILE = BASE_DIR / '.latest_run.txt'
with open(RECORD_FILE, 'w') as f:
    f.write(f"{CURRENT_RUN_NAME}|{task_name}")

print(f"🆔 Run Identifier: {CURRENT_RUN_NAME}")

# ---------------------------------------------------------
# 3. Native Complexa Subprocess Execution
# ---------------------------------------------------------
def run_command_with_env(command_str):
    env = os.environ.copy()
    # Apply strict native environment flags
    env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

    process = subprocess.Popen(
        f"source env.sh && {command_str}",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode

# Construct the exact native command string with dynamic JSON overrides
cmd_str = (
    f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={CURRENT_RUN_NAME} "
    f"++generation.task_name={task_name} ++generation.num_designs={num_designs} "
    f"++generation.dataloader.batch_size={gen_batch_size} ++generation.search.max_batch_size={gen_batch_size} "
    f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
    f"++run_filter={run_filter} ++run_evaluate={run_evaluate} ++run_analyze={run_analyze}"
)

print(f"\n⚙️ Executing standard pipeline via native wrapper...")
ret_code = run_command_with_env(cmd_str)

if ret_code == 0:
    print(f"\n✅ Generation complete! Proceed to the Downstream Analyzer (Cell 7) to extract top candidates.")
else:
    print(f"\n❌ Error: Pipeline crashed (Code {ret_code}). Check logs above.")

# ==============================================================================
# Update Log:
# Purpose: Isolate execution environments by copying the target PDB into a run-specific directory, preventing file IO conflicts during iterative operations.
# Upstream Code: Global configuration dashboard and Section 2 PDB processing.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 08:55:00 EDT
# Changed Lines:
# - Line 5: Imported `shutil` for file system operations.
# - Lines 36-47: Implemented directory generation for `RUN_ISOLATED_DIR` utilizing `CURRENT_RUN_NAME`. Added `shutil.copy` to transfer `SOURCE_PDB_PATH` to the new `ISOLATED_PDB_PATH`.
# - Line 84: Modified the Hydra command string to utilize `ISOLATED_PDB_PATH` instead of the static `FIXED_PDB_PATH`.
# ==============================================================================

In [ ]:
# Cell_6_Intelligent_Error_Log_Analyzer.py
# Requirement: Automatically inspect and diagnose execution logs across the entire pipeline. Retrieve the workload manifest, scan specific log outputs for each phase, and report the exact point of failure. Correctly handle overarching pipeline crashes and fix syntax spacing issues.

import os
import sys
from pathlib import Path

# ==========================================
# 1. Read Task Record (Strictly Inherit Dedicated Variables)
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
LOGS_DIR = BASE_DIR / 'logs'

if not RECORD_FILE.exists():
    print("❌ Error: Record file .latest_run.txt not found. Please execute the pipeline first.")
    sys.exit(1)

content = RECORD_FILE.read_text().strip()
TARGET_RUN_NAME, task_str = content.split('|')
TASKS = task_str.split(',')

print(f"🎯 Retrieving dedicated run code: {TARGET_RUN_NAME}")
print(f"📋 Included target tasks: {TASKS}\n")

# ==========================================
# Intelligent Log Diagnostic Tool
# ==========================================
def check_log_status(log_path):
    """Reads the log to determine success and returns (is_success, log_tail_content)"""
    if not log_path.exists():
        return False, []

    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if not lines:
        return False, ["(Log file is empty)"]

    # Extract only the last 50 lines for analysis and display
    tail_lines = lines[-50:]
    tail_text = "".join(tail_lines)

    # Capture key flag words indicating errors
    error_keywords = [
        "Traceback (most recent call last):",
        "Error:",
        "[ERROR]",
        "OutOfMemoryError",
        "Exception:",
        "failed with exit code",
        "Returned non-zero exit status"
    ]

    for kw in error_keywords:
        if kw in tail_text:
            return False, tail_lines

    # If no error flags are present, evaluate as successful
    return True, tail_lines

# ==========================================
# 2. Iteratively Diagnose Error Logs for Each Target
# ==========================================
for task in TASKS:
    print("="*70)
    print(f"🔎 Diagnosing target [ {task} ]")
    print("="*70)

    # Core matching logic: Compatible with the dynamic long-string nomenclature
    folder_pattern = f"design_pipeline_{task}_{TARGET_RUN_NAME}*"
    matching_folders = sorted(list(LOGS_DIR.glob(folder_pattern)))

    if not matching_folders:
        print(f"❌ Log folder for {task} not found. It may have been deleted or not yet executed.")
        continue

    latest_folder = matching_folders[-1]

    gen_log = latest_folder / 'generate.log'
    filter_log = latest_folder / 'filter.log'
    eval_log = latest_folder / 'evaluate.log'
    main_log = latest_folder.with_name(latest_folder.name + ".log")

    # Check overarching main_log first for pipeline crash
    pipeline_crashed = False
    if main_log.exists():
        main_success, main_lines = check_log_status(main_log)
        if not main_success:
            pipeline_crashed = True
            print(f"❌ Pipeline Level Error (Wrapper Crash): Failed!")
            print(f"🔥 --- Reading {main_log.name} --- 🔥\n")
            print("".join(main_lines))

    # Only inspect sub-logs if the wrapper itself did not report a critical crash
    if not pipeline_crashed:
        # 1. Inspect Generate Phase
        if gen_log.exists():
            gen_success, gen_lines = check_log_status(gen_log)

            if gen_success:
                print("✅ Generate (Generation Phase) : Success!")

                # 2. Inspect Filter Phase
                if filter_log.exists():
                    fil_success, fil_lines = check_log_status(filter_log)
                    if fil_success:
                        print("✅ Filter   (Filtering Phase) : Success!")
                    else:
                        print(f"❌ Filter   (Filtering Phase) : Failed!")
                        print(f"🔥 --- Reading {latest_folder.name}/filter.log --- 🔥\n")
                        print("".join(fil_lines))
                        continue # Halt further inspection if the filtering phase fails

                # 3. Inspect Evaluate Phase
                if eval_log.exists():
                    eval_success, eval_lines = check_log_status(eval_log)
                    if eval_success:
                        print("✅ Evaluate (Evaluation Phase)  : Success!")
                        print("🎉 Conclusion: All executed processes for this target have passed perfectly!\n")
                    else:
                        print(f"❌ Evaluate (Evaluation Phase)  : Failed!")
                        print(f"🔥 --- Reading {latest_folder.name}/evaluate.log --- 🔥\n")
                        print("".join(eval_lines))
                else:
                    print("⚠️ Evaluate (Evaluation Phase)  : evaluate.log not found.")
                    print("   (Note: The current task may have only completed generation and filtering; evaluation was either disabled or not yet reached.)\n")

            else:
                print(f"❌ Generate (Generation Phase) : Failed!")
                print(f"🔥 --- Reading {latest_folder.name}/generate.log --- 🔥\n")
                print("".join(gen_lines))

        else:
            print(f"❌ No step-specific log files found in directory {latest_folder.name}!\n")

# Purpose: Automatically inspect and diagnose execution logs. Rectifies logic flaws where global crash events in main_log were ignored due to elif branch structures, and removes invalid non-breaking spaces causing syntax errors.
# Upstream Code: Original Cell 6 Error Log Analyzer.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 20:15 EDT.
# Changed Lines:
# * Line 10-109: Stripped \xa0 characters and replaced with standard indentation.
# * Line 47-48: Appended system exit keywords ("failed with exit code", "Returned non-zero exit status") to the error detection dictionary.
# * Line 75-82: Refactored main_log evaluation to execute independently prior to step-logs, setting a pipeline_crashed boolean flag to intercept false positives.

In [ ]:
# Cell_7_Deep_Result_Scanner.py
# Requirement: Dynamically retrieve and parse inference results based on the workload manifest.
# Read the target_threshold dynamically from the persistent configuration file to replace hardcoded values.
# Strictly rank generated designs using min_ipSAE, highlight hits, and display auxiliary metrics.

import os
import sys
import csv
import json
from pathlib import Path

# ==========================================
# 1. Load Persistent Configuration & Run Record
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
SETTING_FILE = BASE_DIR / 'internal_settings.json'

# Dynamically load configuration to avoid hardcoded parameters
if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run the setup block first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

# Extract dynamic threshold, fallback to 0.80 if key is missing
TARGET_THRESHOLD = float(cfg.get('target_threshold', 0.80))

if not RECORD_FILE.exists():
    print("❌ Error: Record file .latest_run.txt not found. Please execute the generation pipeline first.")
    sys.exit(1)

content = RECORD_FILE.read_text().strip()
TARGET_RUN_NAME, task_str = content.split('|')
TASKS = task_str.split(',')

print(f"🎯 Retrieving dedicated run code: {TARGET_RUN_NAME}")
print(f"📋 Included target tasks: {TASKS}")
print(f"⚙️ Active screening threshold: {TARGET_THRESHOLD}\n")

# ==========================================
# 2. Iteratively Scan and Parse Results for Each Target
# ==========================================
for task in TASKS:
    print("\n" + "=" * 85)
    print(f"  Target List: [ {task} ]")
    print("=" * 85)

    # Use wildcard matching to accommodate various suffixes added by the underlying framework
    inference_base = BASE_DIR / 'inference'
    folder_pattern = f"*{task}_{TARGET_RUN_NAME}*"
    matching_folders = sorted(list(inference_base.glob(folder_pattern)))

    if not matching_folders:
        print(f"❌ Inference result folder for {task} not found. Generation may be incomplete or an error occurred.")
        continue

    # Retrieve the most recent matching folder
    INFERENCE_DIR = matching_folders[-1]

    all_csv_files = list(INFERENCE_DIR.rglob('*.csv'))
    valid_csv_files = [f for f in all_csv_files if 'timing' not in f.name.lower()]

    if not valid_csv_files:
        print(f"⚠️ No valid CSV result file found in {INFERENCE_DIR.name}.")
        continue

    target_csv = max(valid_csv_files, key=lambda x: x.stat().st_size)

    with open(target_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        print("⚠️ CSV file is empty.")
        continue

    # --- Intelligent Sorting & Extraction Logic ---
    def get_val(row, col):
        try: return float(row.get(col, -999.0))
        except: return -999.0

    # Strictly sort in descending order based on af2folding_min_ipsae
    rows.sort(key=lambda x: get_val(x, 'af2folding_min_ipsae'), reverse=True)

    # --- Print Professional Evaluation Table ---
    header_format = "{:<4} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
    print(header_format.format("Hit", "Design_ID", "ipSAE", "pTM", "iPTM", "pLDDT", "RMSD(Å)", "Reward"))
    print("-" * 85)

    hits_found = 0
    # Display the top 10 designs from this single run
    for i, row in enumerate(rows[:10]):
        # Correct ID: If design_id is empty, fallback to index naming
        d_id = row.get('design_id', '').strip()
        if not d_id: d_id = f"Design_{row.get('pdb_index', i)}"

        # Extract precise metrics
        ipsae = get_val(row, 'af2folding_min_ipsae')
        ptm = get_val(row, 'af2folding_ptm_log')
        iptm = get_val(row, 'af2folding_i_ptm_log')
        plddt = get_val(row, 'af2folding_plddt')
        rmsd = get_val(row, 'af2folding_rmsd')
        reward = get_val(row, 'total_reward')

        # Mark hits visually
        is_hit = ipsae >= TARGET_THRESHOLD
        hit_marker = "🌟" if is_hit else "  "
        if is_hit: hits_found += 1

        # Format and print the row safely
        ipsae_str = f"{ipsae:.4f}" if ipsae != -999.0 else "N/A"
        ptm_str = f"{ptm:.4f}" if ptm != -999.0 else "N/A"
        iptm_str = f"{iptm:.4f}" if iptm != -999.0 else "N/A"
        plddt_str = f"{plddt:.2f}" if plddt != -999.0 else "N/A"
        rmsd_str = f"{rmsd:.2f}" if rmsd != -999.0 else "N/A"
        reward_str = f"{reward:.4f}" if reward != -999.0 else "N/A"

        row_format = "{:<4} | {:<12} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8} | {:<8}"
        print(row_format.format(hit_marker, d_id[:12], ipsae_str, ptm_str, iptm_str, plddt_str, rmsd_str, reward_str))

    print(f"\n✅ Scan Summary: Found {hits_found} 'Hits' (ipSAE >= {TARGET_THRESHOLD}) in the top results.")
    print(f"📂 Source Data Path: {INFERENCE_DIR.relative_to(BASE_DIR)}")

print("\n🎉 Single-Target Leaderboard scan complete. The top-scoring designs are identified above.")

# Purpose: Automate the retrieval and parsing of generated protein designs. Dynamically reads user-defined thresholds instead of hardcoding.
# Upstream Code: Pipeline generation output (Cell 6) and global configuration JSON.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-02 21:28:02 EDT.
# Changed Lines:
# - Line 9: Added `import json`.
# - Lines 18-25: Added logic to locate and parse `internal_settings.json` to dynamically extract the `target_threshold` variable.
# - Line 36: Added a console printout `print(f"⚙️ Active screening threshold: {TARGET_THRESHOLD}\n")` to verify the parsed configuration parameters.

In [ ]:
# # Cell_8_Batch_Packaging_and_Download.py
# # Requirement: Consolidate generated PDB and CSV files across multiple targets into a single staging directory based on the workload manifest, compress them into a unified ZIP archive, and trigger an automatic browser download.

# import os
# import shutil
# import sys
# from pathlib import Path

# try:
#     from google.colab import files
# except ImportError:
#     print("⚠️ Warning: Not executing within a Colab environment. The system will only compress the files without triggering a browser download.")

# # ==========================================
# # 1. Read Task Record (Strictly Inherit Dedicated Variables)
# # ==========================================
# BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
# RECORD_FILE = BASE_DIR / '.latest_run.txt'

# if not RECORD_FILE.exists():
#     print("❌ Error: Record file not found. Unable to determine tasks for packaging. Please execute the generation pipeline first.")
#     sys.exit(1)

# content = RECORD_FILE.read_text().strip()
# TARGET_RUN_NAME, task_str = content.split('|')
# TASKS = task_str.split(',')

# print(f"🎯 Preparing to package dedicated run code: {TARGET_RUN_NAME}")
# print(f"📋 Included tasks: {TASKS}\n")

# # ==========================================
# # 2. Assemble Staging Directory
# # ==========================================
# # Create a temporary directory to consolidate results from all targets
# ARCHIVE_STAGING_DIR = BASE_DIR / f'Batch_Results_Staging_{TARGET_RUN_NAME}'
# ARCHIVE_STAGING_DIR.mkdir(parents=True, exist_ok=True)

# total_pdbs = 0
# total_csvs = 0

# inference_base = BASE_DIR / 'inference'

# for task in TASKS:
#     # Core Modification: Utilize wildcard matching to automatically track extended suffix directories generated by the underlying framework
#     folder_pattern = f"*{task}_{TARGET_RUN_NAME}*"
#     matching_folders = sorted(list(inference_base.glob(folder_pattern)))

#     if matching_folders:
#         # Retrieve the most recent result directory
#         INFERENCE_DIR = matching_folders[-1]

#         pdbs = list(INFERENCE_DIR.rglob('*.pdb'))
#         csvs = list(INFERENCE_DIR.rglob('*.csv'))
#         total_pdbs += len(pdbs)
#         total_csvs += len(csvs)

#         # Copy the entire result directory for this target into the staging area, naming the subdirectory after the task
#         dest_dir = ARCHIVE_STAGING_DIR / task
#         if dest_dir.exists():
#             shutil.rmtree(dest_dir) # Clear pre-existing remnants if any
#         shutil.copytree(INFERENCE_DIR, dest_dir)

#         print(f"✅ Consolidated [ {task} ] into staging (includes {len(pdbs)} PDBs, {len(csvs)} CSVs)")
#     else:
#         print(f"⚠️ Result directory for target [ {task} ] not found (Pattern: {folder_pattern}). Skipped.")

# # ==========================================
# # 3. Compression and Download
# # ==========================================
# if total_pdbs > 0:
#     ZIP_FILE_PATH = BASE_DIR / f'All_Results_{TARGET_RUN_NAME}_Full'
#     print(f"\n📦 Compressing all contents into a single archive. Please wait...")

#     # Execute compression
#     shutil.make_archive(str(ZIP_FILE_PATH), 'zip', ARCHIVE_STAGING_DIR)
#     final_zip = str(ZIP_FILE_PATH) + '.zip'
#     print(f"🎉 Compression complete! Drive backup saved at: {final_zip}")

#     # Clean up the temporary staging directory to maintain storage hygiene
#     shutil.rmtree(ARCHIVE_STAGING_DIR)

#     # Trigger browser download
#     print("⬇️ Requesting browser download. Please monitor for download prompts (check browser popup blockers if none appear)...")
#     try:
#         files.download(final_zip)
#     except NameError:
#         pass
# else:
#     print("\n❌ All directories are empty. No files found to package. Aborting.")
#     # Clean up the empty staging directory
#     if ARCHIVE_STAGING_DIR.exists():
#         shutil.rmtree(ARCHIVE_STAGING_DIR)

# # Purpose: Consolidate generated structures and metrics into a centralized staging area, generate an archive for local analysis, and clear temporary files to maintain Google Drive storage hygiene.
# # Upstream Code: Original Cell 4 (Batch Packaging and Download) from the Proteina_Complexa_coLab.ipynb.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-01 09:10 EDT.
# # Changed Lines:
# # * Renumbered to Cell 8 to maintain sequential logical flow.

In [ ]:
# Cell_9c_Auto_Download_Top_10.py
# Requirement: Automatically extract the top 10 generated designs from the most recent Auto-Pilot master log, package their PDB files, and trigger a browser download without hardcoding filenames.
##################################################
################ download top 10 files############
##################################################
import os, json, csv, shutil
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    print("⚠️ Warning: Not running in Google Colab environment. Files will be packaged but browser download will not trigger.")

BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Pipeline cannot proceed.")
    import sys; sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
master_log_path = BASE_DIR / 'screening_results' / task_name / 'autopilot_master_log.csv'

if not master_log_path.exists():
    print(f"❌ Error: Master log not found at {master_log_path.relative_to(BASE_DIR)}.")
    print("Please ensure the Auto-Pilot Generation Loop has successfully recorded at least one cycle.")
    import sys; sys.exit(1)

print("🔍 Parsing global screening log for top candidates...")

# Read and sanitize the CSV log data
data = []
with open(master_log_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row['min_ipSAE'] = float(row['min_ipSAE'])
            data.append(row)
        except ValueError:
            pass

if not data:
    print("❌ Error: No valid data entries found in the master log.")
    import sys; sys.exit(1)

# Sort descending by min_ipSAE and slice top 10
data.sort(key=lambda x: x['min_ipSAE'], reverse=True)
top_10 = data[:10]

# Establish a temporary staging directory
STAGING_DIR = BASE_DIR / f'Top_10_{task_name}_Staging'
STAGING_DIR.mkdir(parents=True, exist_ok=True)

copied_count = 0
for rank, entry in enumerate(top_10, 1):
    source_dir = BASE_DIR / entry['source_path']
    d_id = entry['design_id']
    score = entry['min_ipSAE']

    target_pdb = None
    if source_dir.exists():
        # Iterate over all PDBs in the source inference directory to locate the matching structure
        for pdb_file in source_dir.rglob('*.pdb'):
            if d_id.lower() in pdb_file.name.lower() or pdb_file.name == f"{d_id}.pdb":
                target_pdb = pdb_file
                break

        # Fallback metric if exact ID is malformed
        if not target_pdb:
            pdbs = list(source_dir.rglob('*.pdb'))
            if pdbs:
                target_pdb = pdbs[0]

    if target_pdb and target_pdb.exists():
        new_name = f"Rank{rank}_{d_id}_score{score:.4f}.pdb"
        shutil.copy(target_pdb, STAGING_DIR / new_name)
        copied_count += 1
        print(f"✅ Packaged Rank {rank}: {new_name}")
    else:
        print(f"⚠️ Could not locate corresponding PDB file for {d_id} in {source_dir.relative_to(BASE_DIR)}")

if copied_count > 0:
    ZIP_PATH = BASE_DIR / 'screening_results' / task_name / f'Top_10_AutoPilot_{task_name}'
    print("\n📦 Compressing top candidates into a ZIP archive...")
    shutil.make_archive(str(ZIP_PATH), 'zip', STAGING_DIR)
    final_zip = str(ZIP_PATH) + '.zip'

    # Clean up the staging directory
    shutil.rmtree(STAGING_DIR)

    print("⬇️ Triggering browser download. Please monitor for download prompts...")
    try:
        files.download(final_zip)
    except NameError:
        pass
else:
    print("\n❌ Failed to copy any valid PDB files. Archive generation aborted.")
    shutil.rmtree(STAGING_DIR)

# Purpose: Automate the extraction, packaging, and downloading of the top 10 generated PDB structures based on the auto-pilot log output.
# Upstream Code: Auto-Pilot Engine (Cell_9b).
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-04 09:37 EDT.
# Changed Lines:
# * This is a completely new cell logic designed to execute independently after Cell 9b completes.

🔍 Parsing global screening log for top candidates...
✅ Packaged Rank 1: Rank1_Design_6_score0.5789.pdb
✅ Packaged Rank 2: Rank2_Design_6_score0.5789.pdb
✅ Packaged Rank 3: Rank3_Design_6_score0.5787.pdb
✅ Packaged Rank 4: Rank4_Design_6_score0.5787.pdb
✅ Packaged Rank 5: Rank5_Design_6_score0.5787.pdb
✅ Packaged Rank 6: Rank6_Design_3_score0.5666.pdb
✅ Packaged Rank 7: Rank7_Design_3_score0.5666.pdb
✅ Packaged Rank 8: Rank8_Design_3_score0.5654.pdb
✅ Packaged Rank 9: Rank9_Design_3_score0.5654.pdb
✅ Packaged Rank 10: Rank10_Design_3_score0.5654.pdb

📦 Compressing top candidates into a ZIP archive...
⬇️ Triggering browser download. Please monitor for download prompts...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Section 4: Spatial Surface Patch Scanning Engine

In [ ]:

# Cell_10_Geometric_Surface_Patch_Scanning_Engine.py
# Requirement: Determine exposed surface residues based on anchored geometrical boundaries and sequentially process designs for derived hotspot patches.
# Implement a robust checkpoint/resume mechanism (断点续传) to survive Colab disconnections. Inject dynamic random seeds per patch to guarantee stochastic structural diversity.

import os, json, sys, subprocess, yaml, textwrap, shutil, datetime, time, random
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.spatial import KDTree, Delaunay

print("🔍 Initializing Geometric Surface Scanning Engine...")

# ---------------------------------------------------------
# 1. Load Persistent Configuration
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please navigate to Section 0 and configure parameters.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
target_chains = cfg['target_chains']
anchor_str = str(cfg['anchor_residues'])
scan_margin = float(cfg['scan_margin'])
local_patch_radius = float(cfg['local_patch_radius'])
designs_per_patch = int(cfg['designs_per_patch'])
batch_size = int(cfg['gen_batch_size'])

anchor_res_list = [int(x.strip()) for x in anchor_str.split(',') if x.strip()]

SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

if not SOURCE_PDB_PATH.exists():
    print(f"❌ Error: Required processed PDB structure not found at {SOURCE_PDB_PATH}. Prior execution of Section 2 is mandatory.")
    sys.exit(1)

SESSION_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
SESSION_NAME = f"GeoScan_{task_name}_{SESSION_TIMESTAMP}"
SESSION_MASTER_DIR = BASE_DIR / 'assets' / 'target_data' / SESSION_NAME
SESSION_MASTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Master Scan Directory established at: {SESSION_MASTER_DIR.relative_to(BASE_DIR)}")

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m}m {s}s"

# ---------------------------------------------------------
# 2. Parse PDB for Spatial Coordinates & Amino Acid Nomenclature
# ---------------------------------------------------------
coords, res_ids, res_names = [], [], []

with open(SOURCE_PDB_PATH, 'r') as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA" and line[21] == target_chains:
            res_names.append(line[17:20].strip())
            res_ids.append(int(line[22:26].strip()))
            coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])

coords, res_ids, res_names = np.array(coords), np.array(res_ids), np.array(res_names)
protein_kdtree = KDTree(coords)

SURFACE_NEIGHBOR_THRESHOLD = 24

# ---------------------------------------------------------
# 3. Anchor Validation & Geometric Convex Hull Construction
# ---------------------------------------------------------
anchor_coords = []
for a_res in anchor_res_list:
    if a_res in res_ids:
        idx = np.where(res_ids == a_res)[0][0]
        neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
        if len(neighbors_10A) > SURFACE_NEIGHBOR_THRESHOLD:
            sys.exit(f"\n❌ FATAL ERROR: Specified Anchor {a_res} ({res_names[idx]}) is mathematically buried within the protein core!")
        anchor_coords.append(coords[idx])
    else:
        sys.exit(f"\n❌ FATAL ERROR: Requested Anchor Residue {a_res} cannot be mapped to target chain {target_chains}.")

if not anchor_coords:
    sys.exit("\n❌ Error: Geometry construction aborted. No validated anchor residues acquired.")

anchor_coords = np.array(anchor_coords)
shape_points = []
spacing = 1.0

n_anchors = len(anchor_coords)
if n_anchors == 1:
    shape_points = anchor_coords
elif n_anchors == 2:
    d = np.linalg.norm(anchor_coords[1] - anchor_coords[0])
    shape_points = np.linspace(anchor_coords[0], anchor_coords[1], max(2, int(d / spacing)))
elif n_anchors == 3:
    d1, d2 = np.linalg.norm(anchor_coords[1] - anchor_coords[0]), np.linalg.norm(anchor_coords[2] - anchor_coords[0])
    for u in np.linspace(0, 1, max(2, int(d1/spacing))):
        for v in np.linspace(0, 1-u, max(2, int(d2/spacing))):
            shape_points.append(anchor_coords[0] + u*(anchor_coords[1]-anchor_coords[0]) + v*(anchor_coords[2]-anchor_coords[0]))
    shape_points = np.array(shape_points)
else:
    hull = Delaunay(anchor_coords, qhull_options='QJ')
    min_b, max_b = np.min(anchor_coords, axis=0), np.max(anchor_coords, axis=0)
    grid_x, grid_y, grid_z = np.mgrid[min_b[0]:max_b[0]:spacing, min_b[1]:max_b[1]:spacing, min_b[2]:max_b[2]:spacing]
    grid_pts = np.vstack((grid_x.ravel(), grid_y.ravel(), grid_z.ravel())).T
    inside = hull.find_simplex(grid_pts) >= 0
    shape_points = grid_pts[inside]

shape_kdtree = KDTree(shape_points if shape_points.ndim == 2 else [shape_points])
candidate_indices = shape_kdtree.query_ball_tree(protein_kdtree, r=scan_margin)
candidate_idx_set = set([idx for sublist in candidate_indices for idx in sublist])

# ---------------------------------------------------------
# 4. Rigorous Topographic Filtration & Resume State Interception (断点续传拦截)
# ---------------------------------------------------------
surface_residues_raw = []
for idx in candidate_idx_set:
    neighbors_10A = protein_kdtree.query_ball_point(coords[idx], r=10.0)
    if len(neighbors_10A) <= SURFACE_NEIGHBOR_THRESHOLD:
        surface_residues_raw.append(idx)

surface_residues = sorted(surface_residues_raw, key=lambda idx: res_ids[idx])

# Define persistent checkpoint file
CHECKPOINT_CSV = BASE_DIR / 'screening_results' / task_name / f'{task_name}_geoscan_checkpoint.csv'
CHECKPOINT_CSV.parent.mkdir(parents=True, exist_ok=True)

all_scan_results = []
completed_res_ids = set()

if CHECKPOINT_CSV.exists():
    try:
        df_checkpoint = pd.read_csv(CHECKPOINT_CSV)
        if not df_checkpoint.empty:
            all_scan_results = df_checkpoint.to_dict('records')
            for val in df_checkpoint['Center_Residue']:
                # Parse format "8 (PHE)" to extract integer 8
                res_id_str = str(val).split('(')[0].strip()
                if res_id_str.isdigit():
                    completed_res_ids.add(int(res_id_str))
            print(f"📥 RESUME STATE DETECTED: Successfully loaded {len(completed_res_ids)} previously processed patch geometries from checkpoint.")
    except Exception as e:
        print(f"⚠️ Warning: Checkpoint file exists but could not be parsed: {e}. Starting fresh.")

# Filter out already completed residues to prevent redundant GPU execution
pending_surface_residues = [idx for idx in surface_residues if res_ids[idx] not in completed_res_ids]

# ---------------------------------------------------------
# 5. Detection Log Formatting
# ---------------------------------------------------------
print("\n" + "="*70)
print(f"🎯 ALGORITHMIC GEOMETRIC SCANNING SECTOR ESTABLISHED")
print(f"Geometry Parameters: {n_anchors} Fixed Anchors | Volumetric Expansion Margin: {scan_margin}Å")
print(f"Total Isolated Candidates: {len(surface_residues)} residues.")
print(f"Pending Execution Queue  : {len(pending_surface_residues)} residues remaining.")
print("-" * 70)

res_strings = [f"{res_ids[idx]}({res_names[idx]})" for idx in pending_surface_residues]
formatted_list = textwrap.fill(", ".join(res_strings), width=80, initial_indent="➤ ", subsequent_indent="  ")
print(formatted_list if res_strings else "➤ Queue Empty. All patches have been scanned.")
print("="*70)

if not pending_surface_residues:
    print("✅ Geometric scanning fully completed based on checkpoint data. Proceeding to final analytics.")
else:
    print("🚀 Queuing generative algorithmic pipelines targeting the pending topographical patches...\n")

# ---------------------------------------------------------
# 6. Core Execution Loop: Patch Isolation & Candidate Synthesis
# ---------------------------------------------------------
env = os.environ.copy()
env.update({'HYDRA_FULL_ERROR': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'PYTHONUNBUFFERED': '1'})

for i, idx in enumerate(pending_surface_residues, 1):
    loop_start_time = time.time()
    center_res = res_ids[idx]
    center_name = res_names[idx]

    local_neighbors = protein_kdtree.query_ball_point(coords[idx], r=local_patch_radius)
    patch_residues = sorted([res_ids[n] for n in local_neighbors])
    patch_str = ",".join(map(str, patch_residues))

    patch_run_name = f"{SESSION_NAME}_Res{center_res}"
    RUN_ISOLATED_DIR = SESSION_MASTER_DIR / patch_run_name
    RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

    ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)

    print(f"\n=======================================================")
    print(f"🔄 [Active Sweep {i}/{len(pending_surface_residues)}] Center Assignment: {center_res} ({center_name})")
    print(f"📂 Local Sandbox: {RUN_ISOLATED_DIR.relative_to(SESSION_MASTER_DIR)}")
    print(f"=======================================================")

    yaml_path = BASE_DIR / 'configs/targets/targets_dict.yaml'
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)

    yaml_data['target_dict_cfg'][task_name]['hotspot_residues'] = [f"{target_chains}{r}" for r in patch_residues]

    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    # Dynamic Random Seed Injection
    dynamic_seed = random.randint(1, 999999)

    cmd_str = (
        f"complexa design configs/search_binder_local_pipeline.yaml ++run_name={patch_run_name} "
        f"++generation.task_name={task_name} ++generation.num_designs={designs_per_patch} "
        f"++generation.dataloader.batch_size={batch_size} ++generation.search.max_batch_size={batch_size} "
        f"++generation.seed={dynamic_seed} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH} "
        f"++run_filter=True ++run_evaluate=False ++run_analyze=False"
    )

    process = subprocess.run(f"source env.sh && {cmd_str}", env=env, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), check=False)
    ret_code = process.returncode

    inference_dir = BASE_DIR / 'inference' / f'search_binder_local_pipeline_{task_name}_{patch_run_name}'

    if ret_code == 0:
        valid_csvs = [f for f in inference_dir.rglob('*.csv') if 'timing' not in f.name.lower()]

        if valid_csvs:
            target_csv = max(valid_csvs, key=lambda x: x.stat().st_size)
            try:
                df = pd.read_csv(target_csv)
                score_col = 'af2folding_min_ipsae' if 'af2folding_min_ipsae' in df.columns else 'total_reward'

                if score_col in df.columns:
                    df_sorted = df.sort_values(by=score_col, ascending=False)
                    top_score = df_sorted[score_col].max()

                    print(f"📝 Evaluation Metrics for Patch {center_res} (Ranked by {score_col}):")
                    print(f"  Rank | Design_ID    | Score")
                    print("  " + "-"*35)

                    for rank, (csv_index, row) in enumerate(df_sorted.iterrows(), 1):
                        d_id = str(row.get('design_id', '')).strip()
                        if not d_id or str(d_id).lower() == 'nan':
                            d_id = f"Design_{row.get('pdb_index', rank)}"
                        s_val = float(row.get(score_col, -999.0))
                        print(f"  {rank:<4} | {d_id[:12]:<12} | {s_val:.4f}")

                    result_entry = {
                        'Center_Residue': f"{center_res} ({center_name})",
                        f'Max_{score_col}': round(top_score, 4),
                        'Designs_Generated': len(df_sorted),
                        'Hotspot_Patch': patch_str
                    }
                    all_scan_results.append(result_entry)

                    # Real-time Checkpointing (实时落盘)
                    pd.DataFrame(all_scan_results).to_csv(CHECKPOINT_CSV, index=False)
                    print(f"💾 Checkpoint Updated: Scan metrics for {center_res} securely written to disk.")

            except Exception as e:
                print(f"⚠️ Data Sync Failure: Unable to parse batch pipeline arrays for {center_res}: {e}")
        else:
            print(f"⚠️ Warning: No valid CSV metrics output located in {inference_dir.name}.")
    else:
        print(f"⚠️ Alert: Generative pipeline exited with non-zero status code {ret_code}. Skipping metric ingestion for {center_res}.")

    cycle_duration = format_time(time.time() - loop_start_time)
    print(f"\n⏱️ Patch Sweep {i} Completed in: {cycle_duration}")

    if i < len(pending_surface_residues):
        import gc
        gc.collect()
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        print("🧹 Buffer memory cleared. Initiating next spatial coordinate...")
        time.sleep(2)

# ---------------------------------------------------------
# 7. Global Analytics and Final Post-Processing Report Output
# ---------------------------------------------------------
if all_scan_results:
    results_df = pd.DataFrame(all_scan_results)

    sort_col = [col for col in results_df.columns if col.startswith('Max_')][0]
    results_df = results_df.sort_values(by=sort_col, ascending=False)

    FINAL_CSV = BASE_DIR / 'screening_results' / task_name / f'geometric_scan_results_FINAL_{SESSION_TIMESTAMP}.csv'
    results_df.to_csv(FINAL_CSV, index=False)

    # Clean up checkpoint after successful completion
    if CHECKPOINT_CSV.exists():
        os.remove(CHECKPOINT_CSV)

    print("\n🏆 ======================================================= 🏆")
    print("           GEOMETRIC SCANNING PROTOCOL CONCLUDED              ")
    print("🏆 ======================================================= 🏆")
    print(f"Top 10 Prioritized Druggable Coordinates (Ranked by Peak Binding Scores):")
    print(results_df[['Center_Residue', sort_col, 'Designs_Generated']].head(10).to_string(index=False))
    print(f"\n📂 Comprehensive analytic report finalized and committed to: {FINAL_CSV}")
else:
    print("\n🛑 Geometric Scanning Protocol Terminated. No viable candidates were successfully generated or evaluated across any surface patches.")

# ==============================================================================
# Update Log:
# Purpose: Implement stateful checkpointing to enable robust resume capabilities after Colab session timeouts. Inject dynamic random seeds to override framework defaults and ensure stochastic generation diversity per patch.
# Upstream Code: Original Cell 10 Geometric Surface Patch Scanning Engine.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 18:46 EDT
# Changed Lines:
# - Line 7: Imported `random` library.
# - Lines 129-145: Engineered the checkpoint interception logic. The engine parses the static `CHECKPOINT_CSV`, extracts integer values from the `Center_Residue` column, and populates `completed_res_ids`.
# - Lines 147-148: Created the `pending_surface_residues` queue, dynamically subtracting already processed spatial patches to bypass redundant execution.
# - Lines 191-192, 196: Injected `dynamic_seed = random.randint(1, 999999)` and embedded `++generation.seed={dynamic_seed}` into the Hydra command line to enforce physical conformation diversity.
# - Lines 231-233: Executed real-time IO disk writes. Appends data to `CHECKPOINT_CSV` immediately after each successful patch evaluation loop.
# - Lines 259-264: Refactored terminal block. Upon full job completion, the engine exports data to a timestamped `FINAL_CSV` and gracefully deletes the temporary `CHECKPOINT_CSV` state file.
# ==============================================================================

# Section 7: Modular Execution: Decoupled Workflow

In [ ]:
# Cell_11a_Pure_Generation_and_Filtering.py
# Requirement: Execute the pure generation and filtering phases by directly invoking the underlying Python modules.
# Bypasses the main complexa wrapper to prevent cascaded execution of evaluation stages.
# Reuses the isolated working directory architecture from the standard pipeline to prevent file access conflicts.

import os, json, sys, subprocess, datetime, shutil, gc, torch
from pathlib import Path

# ---------------------------------------------------------
# 1. Load Persistent Configuration (The "Memory Card")
# ---------------------------------------------------------
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
SETTING_FILE = BASE_DIR / 'internal_settings.json'

if not SETTING_FILE.exists():
    print("❌ Error: Global configuration file not found. Please run Section 0 first.")
    sys.exit(1)

with open(SETTING_FILE, 'r') as f:
    cfg = json.load(f)

task_name = cfg['task_name']
num_designs = cfg['num_designs']
gen_batch_size = cfg['gen_batch_size']
run_filter = cfg['run_filter']

print(f"🚀 Initiating Pure Generation Task for Target: [{task_name}]")
print(f"📊 Designs: {num_designs} | Batch Size: {gen_batch_size}")

# ---------------------------------------------------------
# 2. Path Setup & Run Identification (Isolated Sandbox)
# ---------------------------------------------------------
RUN_TIMESTAMP = datetime.datetime.now().strftime("Y%Y_M%m_D%d_H%H_M%M_S%S")
CURRENT_RUN_NAME = f"Run_{task_name}_PureGen_{RUN_TIMESTAMP}"

# Define the source PDB generated from Section 2
SOURCE_PDB_PATH = BASE_DIR / 'assets' / 'target_data' / task_name / f"{task_name}_fixed.pdb"

# Establish an isolated working directory for the current run
RUN_ISOLATED_DIR = BASE_DIR / 'assets' / 'target_data' / CURRENT_RUN_NAME
RUN_ISOLATED_DIR.mkdir(parents=True, exist_ok=True)

# Define the isolated PDB path and execute the data transfer
ISOLATED_PDB_PATH = RUN_ISOLATED_DIR / f"{task_name}_fixed.pdb"

if SOURCE_PDB_PATH.exists():
    shutil.copy(SOURCE_PDB_PATH, ISOLATED_PDB_PATH)
else:
    print(f"❌ Error: Source PDB not found at {SOURCE_PDB_PATH}. Pipeline cannot proceed.")
    sys.exit(1)

# Save execution manifest for downstream Analyzer or Evaluation cells
RECORD_FILE = BASE_DIR / '.latest_run.txt'
with open(RECORD_FILE, 'w') as f:
    f.write(f"{CURRENT_RUN_NAME}|{task_name}")

print(f"🆔 Run Identifier: {CURRENT_RUN_NAME}")
print(f"📂 Isolated Sandbox: {RUN_ISOLATED_DIR.relative_to(BASE_DIR)}")

# ---------------------------------------------------------
# 3. Decoupled Subprocess Execution
# ---------------------------------------------------------
def run_command_with_env(command_str):
    # Physically clear VRAM
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    env = os.environ.copy()
    # Apply strict native environment flags
    env.update({
        'HYDRA_FULL_ERROR': '1',
        'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
        'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
        'TF_FORCE_GPU_ALLOW_GROWTH': 'true',
        'PYTHONUNBUFFERED': '1'
    })

    process = subprocess.Popen(
        f"source env.sh && {command_str}",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        shell=True,
        executable='/bin/bash',
        cwd=str(BASE_DIR),
        bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode

# Base arguments specifically structured for isolated underlying modules
base_args = (
    "--config-path configs --config-name search_binder_local_pipeline ++job_id=0 "
    f"++run_name={CURRENT_RUN_NAME} ++generation.task_name={task_name} "
    f"++generation.num_designs={num_designs} "
    f"++generation.dataloader.batch_size={gen_batch_size} "
    f"++generation.search.max_batch_size={gen_batch_size} "
    f"++generation.dataloader.dataset.conditional_features.0.pdb_path={ISOLATED_PDB_PATH}"
)

print("\n🔥 [Phase 1/2] Deploying Structural Generative Sequence...")
ret_gen = run_command_with_env(f"python3 -m proteinfoundation.generate {base_args} ++gen_njobs=1")

if ret_gen == 0:
    if run_filter:
        print("\n🔥 [Phase 2/2] Commencing Preliminary Screening Mechanics (Filter)...")
        ret_fil = run_command_with_env(f"python3 -m proteinfoundation.filter {base_args}")
        if ret_fil != 0:
            print(f"⚠️ Warning: Filtering phase encountered an issue (Code {ret_fil}).")
    else:
        print("\n⏭️ Skipping Filtering Phase (Disabled in settings).")
    print("\n🎉 Pure generation processing complete. All associated VRAM has been de-allocated.")
else:
    print(f"\n❌ Error: Generation module crashed (Code {ret_gen}). Check logs above.")


# ==============================================================================
# Update Log:
# Purpose: Refactor the decoupled generation module to inherit the robust directory isolation architecture from Cell 5b.
# Upstream Code: Global configuration dashboard.
# Runtime Environment: Google Colab.
# Generation Time: 2026-04-03 09:15:00 EDT
# Changed Lines:
# - Replaced hardcoded constants with dynamic data extraction from `internal_settings.json`.
# - Implemented `RUN_ISOLATED_DIR` sandbox generation and `ISOLATED_PDB_PATH` file duplication mechanics.
# - Updated `run_command_with_env` execution string interpolation to seamlessly connect `proteinfoundation.generate` and `proteinfoundation.filter` commands to the isolated sandbox.
# ==============================================================================

In [ ]:
# Cell_11b_Automated_Evaluation_and_Analysis.py
# Requirement: Independently execute the evaluation and analysis modules of the protein pipeline.
# Dynamically inherits or manually targets a specific run sandbox to locate generated structures, assess them using predictive oracles (e.g., AlphaFold2), and compile scoring metrics while strictly managing VRAM limits.

import os, subprocess, sys, yaml, gc, torch
from pathlib import Path

# ==========================================
# --- User Configuration Overrides ---
# ==========================================
# To manually evaluate a specific historic run, input the complete run code (e.g., 'Run_GFP_1_10_PureGen_2026...').
# Leave blank to automatically retrieve the latest pipeline execution manifest.
MANUAL_RUN_NAME = ""

# VRAM Optimization: Structural evaluation (AF2) consumes significantly more memory than generation.
# Adjust this batch size based on available GPU limits (e.g., 8 for A100/80GB, 2 for T4/16GB, 4 for L4/24GB).
BATCH_SIZE_EVAL = 4

# ==========================================
# 1. Core Integration Logic & Run Manifest Inheritance
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/Proteina-Complexa')
RECORD_FILE = BASE_DIR / '.latest_run.txt'
YAML_PATH = BASE_DIR / 'configs/targets/targets_dict.yaml'

if MANUAL_RUN_NAME.strip():
    TARGET_RUN_NAME = MANUAL_RUN_NAME.strip()
    # In manual override mode, parse the active YAML configuration to map target tasks
    if YAML_PATH.exists():
        with open(YAML_PATH, 'r') as f:
            TASKS = list(yaml.safe_load(f).get('target_dict_cfg', {}).keys())
    else:
        print("❌ Error: Manual mode requires 'targets_dict.yaml' to resolve the task array, but the file is missing.")
        sys.exit(1)
    print(f"🎯 [Manual Override Active] Orchestrating evaluation for targeted run: {TARGET_RUN_NAME}")
    print(f"📋 Verified Sub-Tasks: {TASKS}")
elif RECORD_FILE.exists():
    content = RECORD_FILE.read_text().strip()
    TARGET_RUN_NAME, task_str = content.split('|')
    TASKS = task_str.split(',')
    print(f"🎯 [Automated Sync] Identified latest execution sandbox: {TARGET_RUN_NAME}")
    print(f"📋 Verified Sub-Tasks: {TASKS}")
else:
    print("❌ Fatal Exception: Execution manifest not found. Deploy the generation module first or utilize MANUAL_RUN_NAME.")
    sys.exit(1)

def run_PF_command(command_str):
    """Encapsulates subprocess initialization alongside aggressive GPU memory scrubbing mechanisms."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    env = os.environ.copy()
    # Apply strict native environment flags for XLA memory management and real-time logging
    env.update({
        'HYDRA_FULL_ERROR': '1',
        'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
        'XLA_PYTHON_CLIENT_ALLOCATOR': 'platform',
        'TF_FORCE_GPU_ALLOW_GROWTH': 'true',
        'PYTHONUNBUFFERED': '1'
    })

    full_command = f"source env.sh && {command_str}"

    process = subprocess.Popen(
        full_command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, shell=True, executable='/bin/bash', cwd=str(BASE_DIR), bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode

# ==========================================
# 2. Iterative Evaluation and Analysis Loop
# ==========================================
for task in TASKS:
    print("\n" + "="*70)
    print(f"📊 Initializing Metric Extraction for Target: [ {task} ]")
    print("="*70)

    # Core Sync: Target the isolated sandbox directory established during the generation phase
    isolated_pdb_path = BASE_DIR / f"assets/target_data/{TARGET_RUN_NAME}/{task}_fixed.pdb"
    expected_inference_dir = BASE_DIR / f"inference/search_binder_local_pipeline_{task}_{TARGET_RUN_NAME}"

    # Pre-Flight Checks: Ensure data exists before firing up AlphaFold2
    if not isolated_pdb_path.exists():
        print(f"⚠️ Warning: Sandbox target PDB not found at {isolated_pdb_path.relative_to(BASE_DIR)}.")

    if not expected_inference_dir.exists():
        print(f"❌ CRITICAL ERROR: The generation folder for this run is missing!")
        print(f"📂 Path checked: {expected_inference_dir.relative_to(BASE_DIR)}")
        print("💡 Diagnosis: The Generation Phase (Cell 11a) likely failed, OR no valid binders passed the preliminary filters.")
        print("🔧 Action: Please re-run the Generation cell and check its logs, or ensure your 'batch_size' and 'designs' are greater than 0.")
        continue # Skip this task safely instead of crashing

    # Construct robust backend parameter strings bypassing the main wrapper
    base_args = (
        f"--config-path {BASE_DIR}/configs --config-name search_binder_local_pipeline ++job_id=0 "
        f"++base_config_name=search_binder_local_pipeline "  # [FIX] Added missing config bridge
        f"++hydra.job.chdir=False "                          # [FIX] Force Hydra to lock absolute paths
        f"++generation.task_name={task} "
        f"++generation.dataloader.dataset.conditional_features.0.pdb_path={isolated_pdb_path} "
        f"++generation.dataloader.batch_size={BATCH_SIZE_EVAL} "
        f"++generation.search.max_batch_size={BATCH_SIZE_EVAL} "
        f"++run_name={TARGET_RUN_NAME} "
    )

    # Execute Independent Evaluation Phase (AF2 Oracle)
    print(f"\n[Operation Phase 1/2] Deploying Structural Predictive Validation Engine ({task})...")
    ret_eval = run_PF_command(f"python3 -m proteinfoundation.evaluate {base_args} ++run_evaluate=True ++eval_njobs=1")

    # Execute Independent Analysis Phase (Quantitative Reporting)
    if ret_eval == 0:
        print(f"\n[Operation Phase 2/2] Generating Quantitative Scoring Datasets ({task})...")
        run_PF_command(f"python3 -m proteinfoundation.analyze {base_args} ++run_analyze=True")
    else:
        print(f"\n⚠️ Alert: Evaluation module returned a non-zero exit code ({ret_eval}). Skipping the Analysis phase to prevent corrupt outputs.")

print("\n🎉 The structural validation loop has concluded. Please utilize the subsequent scanning or packaging modules to extract quantitative parameters.")

🎯 [Automated Sync] Identified latest execution sandbox: Run_GFP_Y2026_M04_D04_H04_M41_S23
📋 Verified Sub-Tasks: ['GFP']

📊 Initializing Metric Extraction for Target: [ GFP ]
❌ CRITICAL ERROR: The generation folder for this run is missing!
📂 Path checked: inference/search_binder_local_pipeline_GFP_Run_GFP_Y2026_M04_D04_H04_M41_S23
💡 Diagnosis: The Generation Phase (Cell 11a) likely failed, OR no valid binders passed the preliminary filters.
🔧 Action: Please re-run the Generation cell and check its logs, or ensure your 'batch_size' and 'designs' are greater than 0.

🎉 The structural validation loop has concluded. Please utilize the subsequent scanning or packaging modules to extract quantitative parameters.


# ⚠️ CRITICAL: Irreversible Environment Eradication & Total Data Purge. Back up Data Before Execution.

In [ ]:
# # forced_cleanup.py


# # ==========================================
# # Forced Environment Cleanup and State Synchronization (Python Native + Terminal Force)
# # ==========================================
# import os
# import shutil

# ROOT_DIR = "/content/drive/MyDrive/Proteina-Complexa"
# BACKUP_DIR = "/content/drive/MyDrive/Proteina_Subfolder_Backups"

# print(">>> Initiating full-path forced cleanup sequence...")

# # 1. Use Terminal commands for physical erasure (handling FUSE mount synchronization)
# print(">>> [1/2] Forcing directory deletion via terminal commands...")
# !rm -rf "{ROOT_DIR}"
# !rm -rf "{BACKUP_DIR}"

# # 2. Use native Python libraries for kernel-state verification
# print(">>> [2/2] Verifying kernel-state via Python...")
# for target_path in [ROOT_DIR, BACKUP_DIR]:
#     if os.path.exists(target_path):
#         print(f"⚠️ Warning: Remnants detected after terminal command. Initiating secondary destruction: {target_path}")
#         try:
#             shutil.rmtree(target_path)
#             print(f"✅ Completely wiped: {target_path}")
#         except Exception as e:
#             print(f"❌ Force deletion failed (please check Drive permissions): {e}")
#     else:
#         print(f"✅ Confirmed path is physically cleared: {target_path}")

# print("\n>>> State refresh complete! The current Google Drive environment is confirmed to be completely clean.")
# print(">>> Please re-run [Cell 3] (Unified Environment Setup) immediately.")

# # Purpose: Completely clear project directories and cache backups using a dual approach of terminal rm -rf and Python's shutil.rmtree to resolve the "ghost folder" issue caused by cloud drive synchronization delays.
# # Upstream Code: User-provided native Python cleanup script.
# # Runtime Environment: Google Colab.
# # Generation Time: 2026-04-01 11:01 EDT.
# # Changed Lines:
# # * Added the !rm -rf terminal command as the first line of defense for cleanup.
# # * Optimized output logs to clearly distinguish between the terminal erasure and kernel verification phases.